In [ ]:
# compute protein-level + segment-level characterization (MODEL ONLY)
# and plot as BOX PLOTS + HISTOGRAMS (no ECDF, no random controls)

from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# 0) Imports from your repo
# -----------------------------
REPO_ROOT = Path("..").resolve()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from segment_characterize import (
    load_residue_assignments,
    CharacterizeConfig,
    StructureConfig,
    StructureCache,
    StructuralMetricConfig,
    characterize_with_random_baseline,
    compute_protein_level_metrics,
    parse_csv_int_list,
)


METHODS = [
    dict(name="PUFFIN (func-aware)",  cluster_dir="../ismb26/segments/puffin_K64_v4/test", prefix="test"),
    dict(name="MinCut (struct-only)", cluster_dir="../ismb26/segments/mincut_K64/test",    prefix="test"),
    dict(name="ESM-Kmeans (unsup)",      cluster_dir="../ismb26/segments/esm_kmeans/test",      prefix="test"),
]

STRUCTURE_DIR = "../data/pdb_chain"
STRUCTURE_FMT = "auto"
PREFER_CIF = True
CONTACT_CUTOFF = 10.0

# IMPORTANT: you asked "do not include random controls" -> keep False
RANDOM_BASELINE = False
RANDOM_SEEDS = (0, 1, 2)

# Downsample for plotting speed
MAX_POINTS_PER_GROUP = 20000
RNG = np.random.default_rng(0)

# Bins for protein partition histogram
PROT_LABEL_BINS = 40

# Size bins for size-binned structural plots (optional)
SIZE_BINS = (5, 10, 15, 20, 25, 30, 40, 60, 80, 120, 200, 999999)

# structural coherence metrics (segment-level)
METRICS = [
    ("cut_ratio",          "Cut ratio (inter / (inter+intra))",          (0, 1.0)),
    ("contact_density",    "Intra-segment contact density",              None),
    ("mean_intra_ca_dist", "Mean intra-segment Cα distance (Å)",          None),
]

# Segment characterization (segment-level)
SEGMENT_METRICS = [
    ("fragmentation_ratio", "Fragmentation ratio (runs / n_res)",        (0, 0.6)),
    ("seq_compactness",     "Sequence compactness (n_res / span)",       (0, 1.05)),
    ("n_res",               "Segment size (# residues)",                 None),
]

# Protein partition metrics
PROTEIN_METRICS_BOX = [
    ("usage_entropy_norm", "Normalized usage entropy"),
    ("boundaries",         "# boundaries (runs − 1)"),
    ("unique_labels",      "# unique labels (segments per protein)"),
]

PROTEIN_METRIC_HIST = ("unique_labels", "# unique labels per protein")

# -----------------------------
# 2) Initialize structure cache + configs
# -----------------------------
struct_cache = None
has_structure = True
smc = StructuralMetricConfig(contact_cutoff=float(CONTACT_CUTOFF))

if STRUCTURE_DIR is not None:
    struct_cache = StructureCache(
        StructureConfig(
            structure_dir=Path(STRUCTURE_DIR),
            fmt=str(STRUCTURE_FMT),
            prefer_cif=bool(PREFER_CIF),
        )
    )
    has_structure = True

cfg = CharacterizeConfig(
    pad_label=-1,
    random_baseline=bool(RANDOM_BASELINE),
    random_seeds=tuple(int(x) for x in RANDOM_SEEDS),
    size_bins=tuple(int(x) for x in SIZE_BINS),
)

# -----------------------------
# 3) Compute protein-level metrics from residue assignments
# -----------------------------
def compute_protein_metrics_from_res_df(res_df: pd.DataFrame, pad_label: int = -1) -> pd.DataFrame:
    rows = []
    for r in res_df.itertuples(index=False):
        pdb_id = str(getattr(r, "pdb_id"))
        labels = parse_csv_int_list(getattr(r, "cluster_ids"))
        if not labels:
            continue
        pm = compute_protein_level_metrics(labels_sorted=labels, pad_label=pad_label)
        pm["pdb_id"] = pdb_id
        rows.append(pm)
    return pd.DataFrame(rows)

# -----------------------------
# 4) Run characterization for each method (MODEL ONLY)
# -----------------------------
protein_rows = []
segment_rows = []

for m in METHODS:
    name = m["name"]
    cluster_dir = Path(m["cluster_dir"])
    prefix = m["prefix"]

    print(f"\n=== {name} ===")
    print(f"Loading: {cluster_dir}/{prefix}_residue_assignments.csv")

    res_df = load_residue_assignments(cluster_dir, prefix)

    # Protein-level metrics
    prot = compute_protein_metrics_from_res_df(res_df, pad_label=cfg.pad_label)
    prot["method"] = name
    protein_rows.append(prot)

    # Segment-level metrics (structural metrics optional via struct_cache)
    out = characterize_with_random_baseline(
        res_df=res_df,
        cfg=cfg,
        struct_cache=struct_cache,
        smc=smc,
    )
    seg = out["segments_model"].copy()
    seg["method"] = name
    segment_rows.append(seg)

protein_data = pd.concat(protein_rows, ignore_index=True)
segment_data = pd.concat(segment_rows, ignore_index=True)

# Clean + downsample segment data for plotting
segment_data["n_res"] = pd.to_numeric(segment_data.get("n_res", np.nan), errors="coerce")
segment_data = segment_data[np.isfinite(segment_data["n_res"]) & (segment_data["n_res"] >= 2)].copy()

def _subsample(g, max_n):
    if len(g) <= max_n:
        return g
    idx = RNG.choice(g.index.to_numpy(), size=max_n, replace=False)
    return g.loc[idx]

segment_data = segment_data.groupby(["method"], group_keys=False).apply(lambda g: _subsample(g, MAX_POINTS_PER_GROUP))

print("\nProtein columns:", sorted(set(protein_data.columns)))
print("Segment columns:", sorted(set(segment_data.columns)))


In [ ]:
protein_data.groupby("method").agg({
    "usage_entropy_norm": ["mean", "std"],
    "usage_gini": ["mean", "std"],
    "boundaries":         ["mean", "std"],
    "unique_labels":      ["mean", "std"],
    "mean_run_length":    ["mean", "std"],
})

In [ ]:
segment_data.groupby("method").agg({
    "cut_ratio":          ["mean", "std"],
    "contact_density":    ["mean", "std"],
    "mean_intra_ca_dist": ["mean", "std", "min", "max"],
    "fragmentation_ratio": ["mean", "std"],
    "seq_compactness":     ["mean", "std"],
    "seq_span":            ["mean", "std"],
    "mean_run_length":     ["mean", "std"],
    "max_run_length":      ["mean", "std"],
    "run_count":          ["mean", "std"],  

}).T

In [ ]:

# -----------------------------
# Plot helpers: 
# -----------------------------
def boxplot_by_method(df: pd.DataFrame, col: str, ylabel: str, title: str, ylim=None, figsize=None):
    d = df.copy()
    methods = list(d["method"].dropna().unique())

    groups = []
    labels = []
    for m in methods:
        x = pd.to_numeric(d[d["method"] == m][col], errors="coerce").dropna().to_numpy(float)
        if x.size >= 10:
            groups.append(x)
            labels.append(m)

    if figsize is None:
        figsize = (max(6.5, 1.2 * len(labels)), 3.2)

    fig = plt.figure(figsize=figsize)
    ax = plt.gca()
    if not groups:
        ax.text(0.5, 0.5, "No data", ha="center", va="center")
    else:
        ax.boxplot(groups, labels=labels, showfliers=False)
        ax.set_xticklabels(labels, rotation=20, ha="right")
        ax.set_ylabel(ylabel)
        if ylim is not None:
            ax.set_ylim(*ylim)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


def hist_by_method(df: pd.DataFrame, col: str, xlabel: str, title: str, bins=50, xlim=None, density=True, logx=False, figsize=(7.2, 3.2)):
    d = df.copy()
    fig = plt.figure(figsize=figsize)
    ax = plt.gca()

    for m in d["method"].dropna().unique():
        x = pd.to_numeric(d[d["method"] == m][col], errors="coerce").dropna().to_numpy(float)
        if logx:
            x = x[x > 0]
        if x.size < 20:
            continue
        ax.hist(
            x,
            bins=bins,
            density=density,
            histtype="stepfilled",
            linewidth=1.6,
            label=m,
        )

    ax.set_xlabel(xlabel)
    ax.set_ylabel("Density" if density else "Count")
    if xlim is not None:
        ax.set_xlim(*xlim)
    if logx:
        ax.set_xscale("log")
    ax.legend(frameon=False, fontsize=9)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


def hist_size_binned_structural(segment_df: pd.DataFrame, metric: str, ylabel: str, title: str,
                               bins=SIZE_BINS, min_per_bin=30):
    """
    Histogram per method per size bin would be too much. Instead:
    - one small-multiple row: one subplot per method
    - within each subplot: histogram of metric for selected bins (collapsed)
    Here we keep it simple: boxplot-by-sizebin is better, but you requested hist/box only.
    We'll do boxplot-by-sizebin (still boxplot) as the cleanest size-controlled view.
    """
    d = segment_df.copy()
    d["n_res_bin"] = pd.cut(d["n_res"], bins=list(bins), include_lowest=True)
    cats = d["n_res_bin"].cat.categories
    bin_labels = [str(c) for c in cats]

    methods = list(d["method"].dropna().unique())
    n = len(methods)

    fig = plt.figure(figsize=(3.8 * n, 3.2))
    for i, m in enumerate(methods, 1):
        ax = fig.add_subplot(1, n, i)
        dd = d[d["method"] == m]

        # prepare boxplot data per bin
        series = []
        labels = []
        for b in cats:
            g = dd[dd["n_res_bin"] == b]
            x = pd.to_numeric(g[metric], errors="coerce").dropna().to_numpy(float)
            if x.size >= min_per_bin:
                series.append(x)
                labels.append(str(b))

        if not series:
            ax.text(0.5, 0.5, "Insufficient data", ha="center", va="center")
        else:
            ax.boxplot(series, labels=labels, showfliers=False)
            ax.set_xticklabels(labels, rotation=40, ha="right")
            ax.set_ylabel(ylabel)
            ax.set_xlabel("Size bin (#res)")
        ax.set_title(m)

    fig.suptitle(title, y=1.05)
    plt.tight_layout()
    plt.show()

#

In [ ]:
# Single figure with subplots: protein partition + segment coherence (+ optional extras)
# Uses ONE matplotlib figure; no separate plt.show() calls per panel.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# Axis-aware plotting helpers
# -----------------------------
def hist_by_method_ax(df, col, xlabel, title, ax, *, bins=50, xlim=None, density=True, logx=False):
    d = df.copy()
    for m in d["method"].dropna().unique():
        x = pd.to_numeric(d[d["method"] == m][col], errors="coerce").dropna().to_numpy(float)
        if logx:
            x = x[x > 0]
        if x.size < 20:
            continue
        ax.hist(
            x,
            bins=bins,
            density=density,
            histtype="step",
            linewidth=1.6,
            label=m,
        )
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Density" if density else "Count")
    if xlim is not None:
        ax.set_xlim(*xlim)
    if logx:
        ax.set_xscale("log")
    ax.set_title(title)
    ax.legend(frameon=False, fontsize=8)


def boxplot_by_method_ax(df, col, ylabel, title, ax, *, ylim=None):
    d = df.copy()
    methods = list(d["method"].dropna().unique())
    groups, labels = [], []
    for m in methods:
        x = pd.to_numeric(d[d["method"] == m][col], errors="coerce").dropna().to_numpy(float)
        if x.size >= 10:
            groups.append(x)
            labels.append(m)

    if not groups:
        ax.text(0.5, 0.5, "No data", ha="center", va="center")
        ax.set_title(title)
        return

    ax.boxplot(groups, labels=labels, showfliers=False)
    ax.set_xticklabels(labels, rotation=20, ha="right")
    ax.set_ylabel(ylabel)
    if ylim is not None:
        ax.set_ylim(*ylim)
    ax.set_title(title)


def size_binned_box_ax(segment_df, metric, ylabel, title, ax, *, bins=SIZE_BINS, min_per_bin=30, method=None):
    """
    One method per axis (prevents overplotting).
    """
    d = segment_df.copy()
    if method is not None:
        d = d[d["method"] == method].copy()

    d["n_res"] = pd.to_numeric(d["n_res"], errors="coerce")
    d = d[np.isfinite(d["n_res"]) & (d["n_res"] >= 2)].copy()
    d["n_res_bin"] = pd.cut(d["n_res"], bins=list(bins), include_lowest=True)

    cats = d["n_res_bin"].cat.categories
    series, labels = [], []
    for b in cats:
        g = d[d["n_res_bin"] == b]
        x = pd.to_numeric(g[metric], errors="coerce").dropna().to_numpy(float)
        if x.size >= min_per_bin:
            series.append(x)
            labels.append(str(b))

    if not series:
        ax.text(0.5, 0.5, "Insufficient data", ha="center", va="center")
        ax.set_title(title)
        return

    ax.boxplot(series, labels=labels, showfliers=False)
    ax.set_xticklabels(labels, rotation=40, ha="right")
    ax.set_ylabel(ylabel)
    ax.set_xlabel("Size bin (#res)")
    ax.set_title(title)


# -----------------------------
# Build a single multi-panel figure
# Layout: 3 rows × 2 cols for the main paper (A–D)
# A: protein partition (hist unique_labels)
# B: segment size (hist n_res)
# C: fragmentation (box)
# D: sequence compactness (hist)
# E: structural coherence cut_ratio by size bin (3 mini-panels: one per method)
# F: mean intra-ca-dist (hist)

fig = plt.figure(figsize=(14, 8))
gs = fig.add_gridspec(3, 2, hspace=0.35, wspace=0.25)

axA = fig.add_subplot(gs[0, 0])
axB = fig.add_subplot(gs[0, 1])
axC = fig.add_subplot(gs[1, 0])
axD = fig.add_subplot(gs[1, 1])
axE_container = fig.add_subplot(gs[2, 0])
#axE_container.axis("off")  
axF = fig.add_subplot(gs[2, 1])

# A) Protein partitioning / granularity
hist_by_method_ax(
    protein_data,
    col="unique_labels",
    xlabel="# unique labels per protein",
    title="A. How methods partition proteins",
    ax=axA,
    bins=PROT_LABEL_BINS,
    density=True,
)

# B) Segment size distribution
hist_by_method_ax(
    segment_data,
    col="n_res",
    xlabel="Segment size (# residues)",
    title="B. Segment size distribution",
    ax=axB,
    bins=60,
    density=True,
)

# C) Sequence fragmentation (segment-level)
boxplot_by_method_ax(
    segment_data,
    col="fragmentation_ratio",
    ylabel="Fragmentation ratio (runs / n_res)",
    title="C. Sequence fragmentation",
    ax=axC,
    ylim=(0, 0.6),
)

hist_by_method_ax(segment_data, "seq_compactness", "Sequence compactness (n_res/span)",
                   "Sequence compactness", axD, bins=60, xlim=(0, 1.05), density=True)

# D) Structural coherence: size-binned cut_ratio (3 small multiples, one per method)
if has_structure and "cut_ratio" in segment_data.columns:
    methods_order = [m["name"] for m in METHODS]

    # create 3 inset axes inside the container cell
    # (manual placement: left, bottom, width, height) in container axis coordinates
    # inset_w = 0.31
    # inset_h = 0.92
    # lefts = [0.00, 0.34, 0.68]
    # for i, meth in enumerate(methods_order):
    #     axEi = axE_container.inset_axes([lefts[i], 0.05, inset_w, inset_h])
    #     size_binned_box_ax(
    #         segment_data,
    #         metric="cut_ratio",
    #         ylabel="Cut ratio" if i == 0 else "",  # avoid repeated y-label clutter
    #         title=f"D{i+1}. {meth}",
    #         ax=axEi,
    #         bins=SIZE_BINS,
    #         min_per_bin=30,
    #         method=meth,
    #     )
    hist_by_method_ax(
        segment_data,
        col="cut_ratio",
        xlabel="Cut ratio",
        title="E. Structural coherence (cut ratio)",
        ax=axE_container,
        bins=60,
        density=True,
    )
else:
    axE_container.text(
        0.5, 0.5,
        "D. Structural coherence unavailable\n(enable STRUCTURE_DIR / check structures)",
        ha="center", va="center"
    )
if has_structure and "mean_intra_ca_dist" in segment_data.columns:
    hist_by_method_ax(
        segment_data,
        col="mean_intra_ca_dist",
        xlabel="Mean intra-segment Cα distance (Å)",
        title="E. Mean intra-segment Cα distance",
        ax=axF,
        bins=60,
        density=True,
    )

fig.suptitle("Protein partitioning and segment coherence", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# Consistent style
# -----------------------------
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.linewidth": 0.8,
})

METHOD_ORDER = [m["name"] for m in METHODS]  # same order everywhere

# Fixed colors (matplotlib default cycle but stable mapping)
COLOR = {
    "PUFFIN (func-aware)":  "C0",
    "MinCut (struct-only)": "C1",
    "ESM-KMeans (unsup)":      "C2",
}

def _get_series(df, method, col):
    x = pd.to_numeric(df[df["method"] == method][col], errors="coerce").dropna().to_numpy(float)
    return x

def _clip_to_quantile(x, q=0.99):
    if x.size == 0:
        return x, None
    hi = np.quantile(x, q)
    return x[x <= hi], hi

def hist_step_ax(df, col, ax, *, xlabel, title, bins=50, density=True, xlim=None, clip_q=None, logx=False):
    for m in METHOD_ORDER:
        if m not in df["method"].unique():
            continue
        x = _get_series(df, m, col)
        if x.size < 20:
            continue

        clip_val = None
        if clip_q is not None:
            x, clip_val = _clip_to_quantile(x, clip_q)

        if logx:
            x = x[x > 0]

        ax.hist(
            x,
            bins=bins,
            density=density,
            histtype="step",
            linewidth=1.6,
            label=m,
            color=COLOR.get(m, None),
        )

    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Density" if density else "Count")
    if xlim is not None:
        ax.set_xlim(*xlim)
    if logx:
        ax.set_xscale("log")
    ax.legend(frameon=False, loc="upper right")


def boxplot_ax(df, col, ax, *, ylabel, title, ylim=None):
    groups, labels, colors = [], [], []
    for m in METHOD_ORDER:
        if m not in df["method"].unique():
            continue
        x = _get_series(df, m, col)
        if x.size >= 10:
            groups.append(x)
            labels.append(m)
            colors.append(COLOR.get(m, "C0"))

    bp = ax.boxplot(groups, labels=labels, showfliers=False, patch_artist=True)
    for patch, c in zip(bp["boxes"], colors):
        patch.set_facecolor("none")
        patch.set_edgecolor(c)
        patch.set_linewidth(1.6)
    for med, c in zip(bp["medians"], colors):
        med.set_color(c)
        med.set_linewidth(2.0)

    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xticklabels(labels) # , rotation=15, ha="right")
    if ylim is not None:
        ax.set_ylim(*ylim)


def size_binned_cutratio_small_multiples(segment_df, ax, *, bins=SIZE_BINS, min_per_bin=30):
    """
    One row of 3 mini-panels inside a single axis area using inset axes.
    This is the fairest presentation for cut_ratio when granularity differs.
    """
    ax.axis("off")
    inset_w, inset_h = 0.31, 0.88
    lefts = [0.00, 0.34, 0.68]

    d = segment_df.copy()
    d["n_res"] = pd.to_numeric(d["n_res"], errors="coerce")
    d = d[np.isfinite(d["n_res"]) & (d["n_res"] >= 2)].copy()
    d["n_res_bin"] = pd.cut(d["n_res"], bins=list(bins), include_lowest=True)
    cats = d["n_res_bin"].cat.categories

    for i, m in enumerate(METHOD_ORDER):
        ax_i = ax.inset_axes([lefts[i], 0.06, inset_w, inset_h])
        dd = d[d["method"] == m].copy()

        series, labels = [], []
        for b in cats:
            g = dd[dd["n_res_bin"] == b]
            x = pd.to_numeric(g["cut_ratio"], errors="coerce").dropna().to_numpy(float)
            if x.size >= min_per_bin:
                series.append(x)
                labels.append(str(b))

        if not series:
            ax_i.text(0.5, 0.5, "Insufficient", ha="center", va="center")
        else:
            bp = ax_i.boxplot(series, labels=labels, showfliers=False, patch_artist=True)
            for patch in bp["boxes"]:
                patch.set_facecolor("none")
                patch.set_edgecolor(COLOR.get(m, "C0"))
                patch.set_linewidth(1.3)
            for med in bp["medians"]:
                med.set_color(COLOR.get(m, "C0"))
                med.set_linewidth(1.8)

            ax_i.set_xticklabels(labels, rotation=40, ha="right")

        ax_i.set_title(m, fontsize=10)
        if i == 0:
            ax_i.set_ylabel("Cut ratio")
        else:
            ax_i.set_ylabel("")
        ax_i.set_xlabel("Size bin")

from scipy.stats import gaussian_kde

def kde_ax(
    df: pd.DataFrame,
    col: str,
    ax,
    *,
    xlabel: str,
    title: str,
    xlim: tuple = None,
    bw_adjust: float = 1.2,
    gridsize: int = 512,
    fill_alpha: float = 0.18,
    clip: tuple = None,   # e.g., (0, 1) for compactness
    logx: bool = False,
    eps: float = 1e-8,
):
    """
    KDE (model-only) with optional shading, plotted on a given axis.
    """
    methods = [m for m in METHOD_ORDER if m in df["method"].unique()]

    # collect data ranges
    xs_min, xs_max = np.inf, -np.inf
    series = {}

    for m in methods:
        x = pd.to_numeric(df[df["method"] == m][col], errors="coerce").dropna().to_numpy(float)
        if x.size < 50:
            continue
        if clip is not None:
            lo, hi = clip
            x = x[(x >= lo) & (x <= hi)]
        if logx:
            x = x[x > 0]
            if x.size < 50:
                continue
            x = np.log10(x + eps)
        series[m] = x
        xs_min = min(xs_min, float(np.min(x)))
        xs_max = max(xs_max, float(np.max(x)))

    if not series:
        ax.text(0.5, 0.5, "No data", ha="center", va="center")
        ax.set_title(title)
        return

    if xlim is not None:
        xs = np.linspace(xlim[0], xlim[1], gridsize)
        ax.set_xlim(*xlim)
    else:
        xs = np.linspace(xs_min, xs_max, gridsize)

    for m, x in series.items():
        kde = gaussian_kde(x)
        kde.set_bandwidth(bw_method=kde.factor * bw_adjust)
        ys = kde(xs)

        ax.plot(xs, ys, linewidth=1.8, color=COLOR[m], label=m)
        ax.fill_between(xs, 0, ys, color=COLOR[m], alpha=fill_alpha)

    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Density")
    if logx:
        ax.set_xscale("log")
        if xlim is not None:
            ax.set_xlim(*xlim)
    ax.legend(frameon=False, fontsize=8)

def size_binned_summary_curve_ax(
    segment_df: pd.DataFrame,
    metric: str,
    ax,
    *,
    bins=SIZE_BINS,
    min_per_bin: int = 50,
    ylabel: str = None,
    title: str = None,
):
    d = segment_df.copy()
    d["n_res"] = pd.to_numeric(d["n_res"], errors="coerce")
    d[metric] = pd.to_numeric(d[metric], errors="coerce")
    d = d[np.isfinite(d["n_res"]) & (d["n_res"] >= 2) & np.isfinite(d[metric])].copy()
    d["bin"] = pd.cut(d["n_res"], bins=list(bins), include_lowest=True)

    # x positions: bin midpoints
    cats = d["bin"].cat.categories
    mids = np.array([(c.left + c.right) / 2 for c in cats], dtype=float)

    for m in METHOD_ORDER:
        dd = d[d["method"] == m]
        med, q25, q75, xs = [], [], [], []
        for i, b in enumerate(cats):
            g = dd[dd["bin"] == b][metric].dropna().to_numpy(float)
            if g.size < min_per_bin:
                continue
            xs.append(mids[i])
            med.append(np.median(g))
            q25.append(np.quantile(g, 0.25))
            q75.append(np.quantile(g, 0.75))

        if len(xs) < 2:
            continue

        xs = np.array(xs)
        med = np.array(med)
        q25 = np.array(q25)
        q75 = np.array(q75)

        ax.plot(xs, med, linewidth=2.0, color=COLOR[m], label=m)
        ax.fill_between(xs, q25, q75, alpha=0.15, color=COLOR[m])

    ax.set_xlabel("Segment size (bin midpoint, # residues)")
    ax.set_ylabel(ylabel or metric)
    ax.set_title(title or f"{metric} (size-controlled)")
    ax.legend(frameon=False, fontsize=8, loc="upper right")


# -----------------------------
# Build improved figure (A–F)
# -----------------------------
fig, axes = plt.subplots(2, 3, figsize=(13.5, 7.6))
(axA, axB, axC), (axD, axE, axF) = axes

# A) Protein partitioning: unique_labels per protein (step hist, smooth bins)
kde_ax(
    protein_data,
    col="unique_labels",
    ax=axA,
    xlabel="# unique labels per protein",
    title="A. How methods partition proteins",
    bw_adjust=1.3,      # slightly smoother for bounded metric
    fill_alpha=0.15,
)
# B) Segment size distribution (step hist)
kde_ax(
    segment_data,
    col="n_res",
    ax=axB,
    xlabel="Segment size (# residues)",
    title="B. Segment size distribution",
    bw_adjust=1.3,      # slightly smoother for bounded metric
    fill_alpha=0.15,
)

# C) Sequence fragmentation (boxplot)
boxplot_ax(
    segment_data,
    col="fragmentation_ratio",
    ax=axC,
    ylabel="Fragmentation ratio (runs / n_res)",
    title="C. Sequence fragmentation",
    ylim=(0, 0.6),
)

# D) Sequence compactness (step hist; zoom near 1 can be misleading; keep full 0–1)
if "seq_compactness" in segment_data.columns:
    kde_ax(
        segment_data,
        col="seq_compactness",
        ax=axD,
        xlabel="Sequence compactness (n_res / span)",
        title="D. Sequence compactness",
        bw_adjust=1.0,      # slightly smoother for bounded metric
        fill_alpha=0.15,
    )
else:
    axD.axis("off")
    axD.text(0.5, 0.5, "seq_compactness not available", ha="center", va="center")

# E) Structural coherence: cut_ratio (SIZE-BINNED small multiples; best practice)
# if has_structure and "cut_ratio" in segment_data.columns:
#     size_binned_cutratio_small_multiples(segment_data, axE, bins=SIZE_BINS, min_per_bin=30)
#     axE.set_title("E. Structural coherence (cut ratio, size-binned)", pad=18)
# else:
#     axE.axis("off")
#     axE.text(0.5, 0.5, "Structural metrics unavailable", ha="center", va="center")

if has_structure and "cut_ratio" in segment_data.columns:
    size_binned_summary_curve_ax(
        segment_data,
        metric="cut_ratio",
        ax=axE,
        bins=SIZE_BINS,
        min_per_bin=80,  # increase for stability
        ylabel="Cut ratio (lower is better)",
        title="E. Structural coherence (cut ratio, size-controlled)",
    )
else:
    axE.axis("off")
    axE.text(0.5, 0.5, "Structural metrics unavailable", ha="center", va="center")


# F) Mean intra-segment Cα distance: heavy tail -> clip at p99 + log x
if has_structure and "mean_intra_ca_dist" in segment_data.columns:
    # clip at 99th percentile and show log scale
    kde_ax(
        segment_data,
        col="mean_intra_ca_dist",
        ax=axF,
        xlabel="Mean intra-segment Cα distance (Å) [log scale, clipped at p99]",
        title="F. Intra-segment Cα distance",
        logx=True,
        bw_adjust=1.0,
        fill_alpha=0.15,
    )
else:
    axF.axis("off")
    axF.text(0.5, 0.5, "mean_intra_ca_dist not available", ha="center", va="center")

fig.suptitle("Protein partitioning and segment coherence", y=0.995, fontsize=14)
plt.tight_layout()
plt.show()


# Granularity of discovered units
Relationship between the number of segments per protein and their typical size.

In [ ]:
# --- build per-protein mean segment size from segment_data ---
seg = segment_data.copy()
seg["n_res"] = pd.to_numeric(seg["n_res"], errors="coerce")
seg = seg[np.isfinite(seg["n_res"])]

# You must have a protein identifier in segment_data; in your script it's typically 'pdb' + 'chain'
# or 'pdb_id'. Adjust ID_COL if needed.
ID_COL = "pdb_id" if "pdb_id" in seg.columns else ("pdb" if "pdb" in seg.columns else None)
if ID_COL is None:
    raise ValueError("segment_data must contain 'pdb_id' or 'pdb' to aggregate per protein.")


# If only 'pdb' exists and you also have 'chain', combine:
if ID_COL == "pdb" and "chain" in seg.columns:
    seg["protein_key"] = seg["pdb"].astype(str) + "_" + seg["chain"].astype(str)
    protein_key = "protein_key"
else:
    protein_key = ID_COL

seg_prot = (
    seg.groupby(["method", protein_key], as_index=False)
       .agg(mean_seg_size=("n_res", "mean"),
            median_seg_size=("n_res", "median"),
            n_segments=("n_res", "size"))
)
# print(seg_prot)
# --- attach protein unique_labels ---
prot = protein_data.copy()

if "pdb_id" in prot.columns:
    prot_key = "pdb_id"
    prot[prot_key] = prot[prot_key].apply(lambda x: x.split('-')[0] + '_' + x.split('_')[1])
elif "pdb" in prot.columns and "chain" in prot.columns:
    prot["protein_key"] = prot["pdb"].astype(str) + "_" + prot["chain"].astype(str)
    prot_key = "protein_key"
else:
    raise ValueError("protein_data must contain 'pdb_id' or ('pdb','chain') to join.")

gran = seg_prot.merge(
    prot[["method", prot_key, "unique_labels"]],
    left_on=["method", protein_key],
    right_on=["method", prot_key],
    how="inner",
)


# sanity
gran["unique_labels"] = pd.to_numeric(gran["unique_labels"], errors="coerce")
gran = gran[np.isfinite(gran["unique_labels"]) & np.isfinite(gran["mean_seg_size"])]
gran

In [ ]:
gran.groupby("method").agg({
    "mean_seg_size": ["mean", "std"],
    "unique_labels": ["mean", "std"],
    "median_seg_size": ["mean", "std"],
    "n_segments":    ["mean", "std"],
})

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def granularity_trend_panel(
    gran: pd.DataFrame,
    ax,
    *,
    xcol="unique_labels",
    ycol="mean_seg_size",   # or "mean_seg_size"
    title="Granularity: segments per protein vs segment size",
    xlabel="# segments per protein",
    ylabel="Mean segment size (# residues)",
    min_n_per_x=25,
    x_clip_q=0.995,
    y_clip_q=0.995,
):
    # robust axis limits
    x_all = pd.to_numeric(gran[xcol], errors="coerce").dropna().to_numpy(float)
    y_all = pd.to_numeric(gran[ycol], errors="coerce").dropna().to_numpy(float)
    x_max = int(np.ceil(np.quantile(x_all, x_clip_q))) if x_all.size else 50
    y_max = float(np.ceil(np.quantile(y_all, y_clip_q))) if y_all.size else 80

    ax.set_xlim(0, max(10, x_max))
    ax.set_ylim(0, max(20, y_max))

    for m in METHOD_ORDER:
        g = gran[gran["method"] == m].copy()
        g[xcol] = pd.to_numeric(g[xcol], errors="coerce")
        g[ycol] = pd.to_numeric(g[ycol], errors="coerce")
        g = g[np.isfinite(g[xcol]) & np.isfinite(g[ycol])]
        ax.scatter(g[xcol], g[ycol], s=5, alpha=0.03, color=COLOR[m], rasterized=True)

        # group by integer x (segments/protein)
        xs, med, q25, q75 = [], [], [], []
        for xval, gg in g.groupby(g[xcol].astype(int)):
            if len(gg) < min_n_per_x:
                continue
            ys = gg[ycol].to_numpy(float)
            xs.append(int(xval))
            med.append(np.median(ys))
            q25.append(np.quantile(ys, 0.25))
            q75.append(np.quantile(ys, 0.75))

        if len(xs) < 3:
            continue

        xs = np.array(xs)
        order = np.argsort(xs)
        xs = xs[order]
        med = np.array(med)[order]
        q25 = np.array(q25)[order]
        q75 = np.array(q75)[order]

        ax.plot(xs, med, lw=2.2, color=COLOR[m], label=m)
        ax.fill_between(xs, q25, q75, color=COLOR[m], alpha=0.18, linewidth=0)

    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.18)
    ax.legend(frameon=False, fontsize=9, loc="upper right")


# ---- Usage (single panel) ----
fig, ax = plt.subplots(1, 1, figsize=(6.6, 3.6))
granularity_trend_panel(
    gran,
    ax,
    ycol="median_seg_size",      # <- best default
    title="Granularity: segments per protein vs median segment size",
    xlabel="# segments per protein",
    ylabel="Median segment size (# residues)",
    min_n_per_x=10,              # stability
)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

# -----------------------------
# Consistent style (cleaner)
# -----------------------------
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.linewidth": 0.8,
})

METHOD_ORDER = [m["name"] for m in METHODS]  # same order everywhere

COLOR = {
    "PUFFIN (func-aware)":  "C0",
    "MinCut (struct-only)": "C1",
    "ESM-KMeans (unsup)":      "C2",
}

def _get_series(df, method, col):
    x = pd.to_numeric(df[df["method"] == method][col], errors="coerce").dropna().to_numpy(float)
    return x

def _panel_label(ax, s):
    ax.text(
        0.02, 0.98, s,
        transform=ax.transAxes,
        ha="left", va="top",
        fontsize=11, fontweight="bold"
    )

def _beautify(ax, grid_alpha=0.18):
    ax.grid(True, alpha=grid_alpha)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# -----------------------------
# Boxplot (same logic, nicer ticks)
# -----------------------------
def boxplot_ax(df, col, ax, *, ylabel, title, ylim=None):
    groups, labels, colors = [], [], []
    for m in METHOD_ORDER:
        if m not in df["method"].unique():
            continue
        x = _get_series(df, m, col)
        if x.size >= 10:
            groups.append(x)
            labels.append(m)
            colors.append(COLOR.get(m, "C0"))

    bp = ax.boxplot(groups, labels=labels, showfliers=False, patch_artist=True, widths=0.55)
    for patch, c in zip(bp["boxes"], colors):
        patch.set_facecolor("none")
        patch.set_edgecolor(c)
        patch.set_linewidth(1.6)
    for whisk, c in zip(bp["whiskers"], np.repeat(colors, 2)):
        whisk.set_color(c)
        whisk.set_linewidth(1.2)
    for cap, c in zip(bp["caps"], np.repeat(colors, 2)):
        cap.set_color(c)
        cap.set_linewidth(1.2)
    for med, c in zip(bp["medians"], colors):
        med.set_color(c)
        med.set_linewidth(2.0)

    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xticklabels(labels, rotation=15, ha="right")
    if ylim is not None:
        ax.set_ylim(*ylim)
    _beautify(ax)

# -----------------------------
# Correct log-KDE (KDE in log space, plotted on log-x with Jacobian)
# -----------------------------
def kde_logx_ax(
    df: pd.DataFrame,
    col: str,
    ax,
    *,
    xlabel: str,
    title: str,
    clip_q: float = 0.99,
    bw_adjust: float = 1.0,
    gridsize: int = 512,
    fill_alpha: float = 0.14,
    eps: float = 1e-9,
):
    """
    Computes KDE over z=log10(x) and converts to density over x.
    If f_Z(z) is density in z, then f_X(x)=f_Z(log10 x) * 1/(x ln 10).
    """
    methods = [m for m in METHOD_ORDER if m in df["method"].unique()]
    series = {}

    # gather and clip
    for m in methods:
        x = pd.to_numeric(df[df["method"] == m][col], errors="coerce").dropna().to_numpy(float)
        x = x[np.isfinite(x) & (x > 0)]
        if x.size < 80:
            continue
        hi = np.quantile(x, clip_q) if clip_q is not None else None
        if hi is not None:
            x = x[x <= hi]
        if x.size < 80:
            continue
        series[m] = x

    if not series:
        ax.text(0.5, 0.5, "No data", ha="center", va="center")
        ax.set_title(title)
        return

    # shared grid in log10-space
    all_x = np.concatenate(list(series.values()), axis=0)
    zmin = np.log10(np.min(all_x) + eps)
    zmax = np.log10(np.max(all_x) + eps)
    zs = np.linspace(zmin, zmax, gridsize)
    xs = 10 ** zs  # back to x for plotting

    for m, x in series.items():
        z = np.log10(x + eps)
        kde = gaussian_kde(z)
        kde.set_bandwidth(bw_method=kde.factor * bw_adjust)

        fz = kde(zs)                            # density in z
        fx = fz / (xs * np.log(10.0) + eps)     # convert to density in x

        ax.plot(xs, fx, linewidth=2.0, color=COLOR[m], label=m)
        ax.fill_between(xs, 0, fx, color=COLOR[m], alpha=fill_alpha, linewidth=0)

    ax.set_xscale("log")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Density")
    _beautify(ax, grid_alpha=0.16)

def boxplot_logx_ax(
    df: pd.DataFrame,
    col: str,
    ax,
    *,
    xlabel: str,
    title: str,
    clip_q: float = 0.99,
    min_n: int = 80,
):
    groups, labels, colors = [], [], []

    for m in METHOD_ORDER:
        if m not in df["method"].unique():
            continue
        x = pd.to_numeric(df[df["method"] == m][col], errors="coerce").dropna().to_numpy(float)
        x = x[np.isfinite(x) & (x > 0)]
        if x.size < min_n:
            continue
        if clip_q is not None:
            hi = np.quantile(x, clip_q)
            x = x[x <= hi]
        if x.size < min_n:
            continue

        groups.append(x)
        labels.append(m)
        colors.append(COLOR.get(m, "C0"))

    if not groups:
        ax.text(0.5, 0.5, "No data", ha="center", va="center")
        ax.set_title(title)
        return

    bp = ax.boxplot(groups, labels=labels, showfliers=False, patch_artist=True, widths=0.55)

    for patch, c in zip(bp["boxes"], colors):
        patch.set_facecolor("none")
        patch.set_edgecolor(c)
        patch.set_linewidth(1.6)
    for whisk, c in zip(bp["whiskers"], np.repeat(colors, 2)):
        whisk.set_color(c); whisk.set_linewidth(1.2)
    for cap, c in zip(bp["caps"], np.repeat(colors, 2)):
        cap.set_color(c); cap.set_linewidth(1.2)
    for med, c in zip(bp["medians"], colors):
        med.set_color(c); med.set_linewidth(2.0)

    ax.set_xscale("log")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Mean intra-segment Cα distance (Å)")
    ax.set_xticklabels(labels, rotation=15, ha="right")

    ax.grid(True, alpha=0.16)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

def boxplot_logy_ax(
    df: pd.DataFrame,
    col: str,
    ax,
    *,
    ylabel: str,
    title: str,
    clip_q: float = 0.99,
    min_n: int = 80,
    ylim: tuple = None,
):
    groups, labels, colors = [], [], []

    for m in METHOD_ORDER:
        if m not in df["method"].unique():
            continue

        x = pd.to_numeric(df[df["method"] == m][col], errors="coerce").dropna().to_numpy(float)
        x = x[np.isfinite(x) & (x > 0)]
        if x.size < min_n:
            continue

        if clip_q is not None:
            hi = np.quantile(x, clip_q)
            x = x[x <= hi]

        if x.size < min_n:
            continue

        groups.append(x)
        labels.append(m)
        colors.append(COLOR.get(m, "C0"))

    if not groups:
        ax.text(0.5, 0.5, "No data", ha="center", va="center")
        ax.set_title(title)
        return

    bp = ax.boxplot(groups, labels=labels, showfliers=False, patch_artist=True, widths=0.55)

    for patch, c in zip(bp["boxes"], colors):
        patch.set_facecolor("none")
        patch.set_edgecolor(c)
        patch.set_linewidth(1.6)
    for whisk, c in zip(bp["whiskers"], np.repeat(colors, 2)):
        whisk.set_color(c); whisk.set_linewidth(1.2)
    for cap, c in zip(bp["caps"], np.repeat(colors, 2)):
        cap.set_color(c); cap.set_linewidth(1.2)
    for med, c in zip(bp["medians"], colors):
        med.set_color(c); med.set_linewidth(2.0)

    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_yscale("log")  
    ax.set_xticklabels(labels, rotation=15, ha="right")

    if ylim is not None:
        ax.set_ylim(*ylim)

    ax.grid(True, which="both", axis="y", alpha=0.15)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def quantile_interval_logx_ax(
    df: pd.DataFrame,
    col: str,
    ax,
    *,
    title: str,
    xlabel: str,
    clip_q: float = 0.99,
    qs=(0.10, 0.25, 0.50, 0.75, 0.90),
    min_n: int = 80,
):
    methods = [m for m in METHOD_ORDER if m in df["method"].unique()]
    y = np.arange(len(methods))[::-1]  # top to bottom

    for yi, m in zip(y, methods):
        x = pd.to_numeric(df[df["method"] == m][col], errors="coerce").dropna().to_numpy(float)
        x = x[np.isfinite(x) & (x > 0)]
        if x.size < min_n:
            continue
        if clip_q is not None:
            x = x[x <= np.quantile(x, clip_q)]
        if x.size < min_n:
            continue

        q10, q25, q50, q75, q90 = np.quantile(x, qs)
        c = COLOR[m]

        ax.plot([q10, q90], [yi, yi], color=c, lw=1.4, alpha=0.9)      # 10–90
        ax.plot([q25, q75], [yi, yi], color=c, lw=4.0, alpha=0.9)      # IQR
        ax.plot([q50], [yi], marker="o", color=c, markersize=6)        # median

    ax.set_yticks(y)
    ax.set_yticklabels(methods)
    ax.set_xscale("log")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("")
    ax.grid(True, axis="x", alpha=0.16)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)



# --- quantile-bars panel (log-x, no weird boxplot scaling) --------------------
def quantile_bars_ax(
    df: pd.DataFrame,
    col: str,
    ax,
    *,
    title: str,
    xlabel: str,
    clip_q: float = 0.99,
    qs=(0.10, 0.25, 0.50, 0.75, 0.90),
    min_n: int = 80,
    ytick_fontsize: int = 9,
):
    """
    For each method:
      - thin bar: q10–q90
      - thick bar: q25–q75 (IQR)
      - dot: median (q50)
    X-axis is log-scale (distance in Å). Y-axis is categorical methods.
    """
    methods = [m for m in METHOD_ORDER if m in df["method"].unique()]
    if not methods:
        ax.axis("off")
        ax.text(0.5, 0.5, "No data", ha="center", va="center")
        return

    # plot order top->bottom
    y = np.arange(len(methods))[::-1]

    for yi, m in zip(y, methods):
        x = pd.to_numeric(df[df["method"] == m][col], errors="coerce").dropna().to_numpy(float)
        x = x[np.isfinite(x) & (x > 0)]
        if x.size < min_n:
            continue

        if clip_q is not None:
            hi = np.quantile(x, clip_q)
            x = x[x <= hi]
        if x.size < min_n:
            continue

        q10, q25, q50, q75, q90 = np.quantile(x, qs)
        print(f"{m}: n={x.size}, q10={q10:.2f}, q25={q25:.2f}, q50={q50:.2f}, q75={q75:.2f}, q90={q90:.2f}")
        c = COLOR.get(m, "C0")

        # 10–90 whisker
        ax.plot([q10, q90], [yi, yi], color=c, lw=1.6, alpha=0.95, solid_capstyle="round")
        # IQR
        ax.plot([q25, q75], [yi, yi], color=c, lw=6.0, alpha=0.95, solid_capstyle="round")
        # median
        ax.plot([q50], [yi], marker="o", color="white", markersize=6.5, markeredgecolor=c, markeredgewidth=1.6)
        ax.plot([q50], [yi], marker="o", color=c, markersize=3.2)

    ax.set_yticks(y)
    ax.set_yticklabels(methods, fontsize=ytick_fontsize)
    #ax.set_xscale("log")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("")

    # cosmetics
    ax.grid(True, axis="x", alpha=0.18)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # helpful: force a bit of padding so bars don't touch frame
    xmin, xmax = ax.get_xlim()
    print(xmax)
    ax.set_xlim(xmin * 0.9, xmax * 1.1)
    # ax.set_xticks([1, 10, 100, 400])
    # ax.set_xticklabels(["1", "10", "100", "400"])





# -----------------------------
# Build improved 1x3 figure: (A) fragmentation, (B) cut_ratio size-controlled, (C) intra Cα distance
# -----------------------------
# Wider middle panel
fig = plt.figure(figsize=(10.2, 3.6))
gs = fig.add_gridspec(1, 3, width_ratios=[1.05, 1.55, 1.10], wspace=0.32)

axA = fig.add_subplot(gs[0, 0])
axB = fig.add_subplot(gs[0, 1])
axC = fig.add_subplot(gs[0, 2])

# A) Sequence fragmentation
boxplot_ax(
    segment_data,
    col="fragmentation_ratio",
    ax=axA,
    ylabel="Fragmentation ratio (runs / n_res)",
    title="Sequence fragmentation",
    ylim=(0, 0.6),
)
_panel_label(axA, "A")

# B) Structural coherence (size-controlled cut ratio)
if has_structure and "cut_ratio" in segment_data.columns:
    size_binned_summary_curve_ax(
        segment_data,
        metric="cut_ratio",
        ax=axB,
        bins=SIZE_BINS,
        min_per_bin=80,
        ylabel="Cut ratio (lower is better)",
        title="Structural coherence (size-controlled)",
    )
    _panel_label(axB, "B")
else:
    axB.axis("off")
    axB.text(0.5, 0.5, "Structural metrics unavailable", ha="center", va="center")

# C) Intra-segment Cα distance (log-x KDE, clipped)
# if has_structure and "mean_intra_ca_dist" in segment_data.columns:
#     kde_logx_ax(
#         segment_data,
#         col="mean_intra_ca_dist",
#         ax=axC,
#         xlabel="Mean intra-segment Cα distance (Å) [log x, clipped p99]",
#         title="Intra-segment compactness",
#         clip_q=0.99,
#         bw_adjust=1.0,
#         fill_alpha=0.12,
#     )
#     _panel_label(axC, "C")
# else:
#     axC.axis("off")
#     axC.text(0.5, 0.5, "mean_intra_ca_dist not available", ha="center", va="center")
# if has_structure and "mean_intra_ca_dist" in segment_data.columns:
#     boxplot_logx_ax(
#         segment_data,
#         col="mean_intra_ca_dist",
#         ax=axC,
#         xlabel="(log scale, clipped at p99)",
#         title="Intra-segment compactness",
#         clip_q=0.99,
#     )
#     _panel_label(axC, "C")

# quantile_interval_logx_ax(
#     segment_data,
#     col="mean_intra_ca_dist",
#     ax=axC,
#     title="Intra-segment compactness",
#     xlabel="Mean intra-segment Cα distance (Å) [log x, clipped p99]",
#     clip_q=0.99,
# )
# _panel_label(axC, "C")

# boxplot_logy_ax(
#     segment_data,
#     col="mean_intra_ca_dist",
#     ax=axC,
#     ylabel="Mean intra-segment Cα distance (Å) [log y, clipped p99]",
#     title="Intra-segment compactness",
#     clip_q=0.99,
# )
# _panel_label(axC, "C")


quantile_bars_ax(
    segment_data,
    col="mean_intra_ca_dist",
    ax=axC,
    title="C. Intra-segment compactness",
    xlabel="Mean intra-segment Cα distance (Å) [log x, clipped p99]",
    clip_q=1, # 0.99,
    min_n=1, 
)

_panel_label(axC, "C")

# One shared legend (clean)
handles, labels = axB.get_legend_handles_labels()
if handles:
    fig.legend(handles, labels, loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.53, 1.03))

fig.suptitle("Protein partitioning and segment coherence", y=1.10, fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
fig = plt.figure(figsize=(10.4, 7.2))
gs = fig.add_gridspec(2, 3, height_ratios=[1.05, 1.00], wspace=0.28, hspace=0.35)

axG = fig.add_subplot(gs[0, :])   # top row spans both columns
axA = fig.add_subplot(gs[1, 0])
axB = fig.add_subplot(gs[1, 1])
axC = fig.add_subplot(gs[1, 2])


# Top: granularity trend
granularity_trend_panel(
    gran,
    axG,
    ycol="median_seg_size",
    title="Granularity: segments per protein vs median segment size",
    xlabel="# segments per protein",
    ylabel="Median segment size (# residues)",
    min_n_per_x=10,
)
_panel_label(axG, "A")

# Bottom-left: fragmentation
boxplot_ax(
    segment_data,
    col="fragmentation_ratio",
    ax=axA,
    ylabel="Fragmentation ratio (runs / n_res)",
    title="Sequence fragmentation",
    ylim=(0, 0.6),
)
_panel_label(axA, "B")

# Bottom-right: choose structural coherence OR compactness (pick one)

if has_structure and "cut_ratio" in segment_data.columns:
    size_binned_summary_curve_ax(
        segment_data,
        metric="cut_ratio",
        ax=axB,
        bins=SIZE_BINS,
        min_per_bin=80,
        ylabel="Cut ratio (lower is better)",
        title="Structural coherence (size-controlled)",
    )
    _panel_label(axB, "C")
else:
    axB.axis("off")
    axB.text(0.5, 0.5, "Structural metrics unavailable", ha="center", va="center")

if has_structure and "mean_intra_ca_dist" in segment_data.columns:
    quantile_bars_ax(
        segment_data,
        col="mean_intra_ca_dist",
        ax=axC,
        title="Intra-segment compactness",
        xlabel="Mean intra-segment Cα distance (Å)",
        min_n=80,
    )
    _panel_label(axC, "D")


fig.suptitle("Protein partitioning and segment coherence", y=0.98, fontsize=14)
handles, labels = axA.get_legend_handles_labels()
if handles:
    fig.legend(handles, labels, loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.01))
# set off y tick labels off
#axA.set_xticks([]) 
axC.set_yticks([])
plt.tight_layout()
plt.show()

# Effect of functional supervision

In [ ]:
# --- Q4 kNN neighborhood semantic coherence: 
# Overlaid distributions + violin/box per method, with mean + bootstrap CI.

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================
# CONFIG
# =========================
SPLIT = "valid"
GO_ASPECT = "MF"  # just for titles; CSV already encodes what you ran
REPORTS_ROOT = Path("../ismb26/results/segment_func_reports")  # <-- adjust if needed
OUT_DIR = REPORTS_ROOT / "_figures" / f"knn_q4_{SPLIT}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Your story methods (add/remove as needed)
METHODS = {
    "puffin": "puffin_K64_v4",
    "esm-kmeans": "esm-kmeans",
    "mincut": "mincut_K64"
    # "controls_only": "some_control_model_name_if_you_have_one",  # optional
}

# Tags expected in per_query CSV (from your script)
TAGS_ORDER = ["knn_true", "control_shuffled_embeddings", "control_random_neighbors"]
TAG_PRETTY = {
    "knn_true": "true kNN",
    "control_shuffled_embeddings": "shuffled embeddings",
    "control_random_neighbors": "random neighbors",
}

# --- Metrics to show (edit as you like) ---
# "Neighborhood GO enrichment": use -log10(best_pval) (higher is better), optionally clipped.
# "Neighborhood purity / F-score-like": shared_go_frac_neighbors (higher better) + hit_any_shared_go (binary)
METRICS = [
    ("shared_go_frac_neighbors", "Neighborhood purity (shared GO fraction)", dict(xlim=(0, 1))),
    ("hit_any_shared_go", "Hit@k (any neighbor shares ≥1 GO term)", dict(xlim=(0, 1))),
    ("neglog10_best_pval", "Neighborhood enrichment (-log10 best Fisher p)", dict(xlim=(0, None))),
    ("go_entropy_neighborhood", "Neighborhood GO entropy (normalized; lower = more coherent)", dict(xlim=(0, 1))),
]

# bootstrap CI settings
BOOT_N = 2000
CI = 0.95
RNG_SEED = 0

# =========================
# HELPERS
# =========================
def bootstrap_mean_ci(x, n_boot=2000, ci=0.95, seed=0):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    means = []
    for _ in range(int(n_boot)):
        samp = rng.choice(x, size=x.size, replace=True)
        means.append(np.mean(samp))
    means = np.asarray(means)
    alpha = (1.0 - ci) / 2.0
    lo = np.quantile(means, alpha)
    hi = np.quantile(means, 1.0 - alpha)
    return float(np.mean(x)), float(lo), float(hi)

def load_one_method(method_name, model_dirname, split):
    p = REPORTS_ROOT / model_dirname / split / f"unit_knn_go_enrichment_{split}_per_query.csv"
    if not p.exists():
        raise FileNotFoundError(f"Missing per_query CSV: {p}")
    df = pd.read_csv(p)

    # keep only the tags we want (sometimes extra tags exist)
    df = df[df["tag"].isin(TAGS_ORDER)].copy()

    # Standardize
    df["method"] = method_name
    df["tag_pretty"] = df["tag"].map(TAG_PRETTY).fillna(df["tag"].astype(str))

    # Build -log10(best_pval) as enrichment strength; keep NaNs if no best term.
    bp = pd.to_numeric(df.get("best_pval", np.nan), errors="coerce").astype(float)
    # avoid inf: clamp extremely tiny pvals
    bp = np.clip(bp, 1e-300, 1.0)
    df["neglog10_best_pval"] = -np.log10(bp)
    # if best_pval was NaN, neglog10 will be NaN (fine)
    df.loc[~np.isfinite(pd.to_numeric(df.get("best_pval", np.nan), errors="coerce")), "neglog10_best_pval"] = np.nan

    # Optional: restrict to queries that have GO (your script already sets has_go)
    if "has_go" in df.columns:
        df = df[df["has_go"] == 1].copy()

    return df

def savefig_dual(fig, out_base: Path):
    fig.savefig(out_base.with_suffix(".png"), dpi=250, bbox_inches="tight")
    fig.savefig(out_base.with_suffix(".pdf"), bbox_inches="tight")
    plt.close(fig)

# =========================
# LOAD DATA
# =========================
dfs = []
for m_pretty, m_dir in METHODS.items():
    dfs.append(load_one_method(m_pretty, m_dir, SPLIT))
df_all = pd.concat(dfs, ignore_index=True)

# Ensure consistent ordering
df_all["tag"] = pd.Categorical(df_all["tag"], categories=TAGS_ORDER, ordered=True)
df_all["method"] = pd.Categorical(df_all["method"], categories=list(METHODS.keys()), ordered=True)

print("Loaded rows:", len(df_all))
print(df_all.groupby(["method", "tag"]).size().unstack(fill_value=0))

# =========================
# (A) STRONGEST: OVERLAID DISTRIBUTIONS (per method, per metric) + mean + CI
# =========================
for metric, title, opts in METRICS:
    fig = plt.figure(figsize=(6, 4 * len(METHODS)))
    for r, method in enumerate(df_all["method"].cat.categories, start=1):
        ax = fig.add_subplot(len(METHODS), 1, r)

        # overlay tag distributions
        for tag in TAGS_ORDER:
            x = df_all[(df_all["method"] == method) & (df_all["tag"] == tag)][metric].to_numpy(dtype=float)
            x = x[np.isfinite(x)]
            if x.size == 0:
                continue

            # density histogram overlay (matplotlib default cycle handles colors)
            ax.hist(x, bins=50, density=True, alpha=0.35, label=TAG_PRETTY.get(tag, tag))

            # mean + CI
            mu, lo, hi = bootstrap_mean_ci(x, n_boot=BOOT_N, ci=CI, seed=RNG_SEED)
            ax.axvline(mu, linestyle="--", linewidth=1.5)
            ax.axvspan(lo, hi, alpha=0.12)

        ax.set_title(f"{method} — {title}")
        ax.set_ylabel("Density")
        ax.legend(loc="best", frameon=True)

        # x-limits if provided
        xlim = opts.get("xlim", None)
        if xlim is not None:
            lo, hi = xlim
            if hi is None:
                # auto upper bound from data
                hi = np.nanpercentile(df_all[df_all["method"] == method][metric].astype(float), 99.5)
                if not np.isfinite(hi):
                    hi = None
            ax.set_xlim(lo, hi)

    fig.tight_layout()
    #plt.show()
    # savefig_dual(fig, OUT_DIR / f"overlay_{metric}_{SPLIT}")

print(f"[OK] Wrote overlay plots to: {OUT_DIR}")

# =========================
# (B) BOX + VIOLIN PER METHOD (one figure per metric)
# =========================
for metric, title, opts in METRICS:
    fig = plt.figure(figsize=(6, 2))
    ax = fig.add_subplot(111)

    # Build grouped data: for each (method, tag)
    groups = []
    xticks = []
    for method in df_all["method"].cat.categories:
        for tag in TAGS_ORDER:
            x = df_all[(df_all["method"] == method) & (df_all["tag"] == tag)][metric].to_numpy(dtype=float)
            x = x[np.isfinite(x)]
            groups.append(x)
            xticks.append(f"{method}\n{TAG_PRETTY.get(tag, tag)}")

    pos = np.arange(1, len(groups) + 1)

    # Violin + box
    ax.violinplot(groups, positions=pos, showmeans=False, showmedians=True, showextrema=False)
    ax.boxplot(groups, positions=pos, widths=0.18, showfliers=False)

    # Mean + CI markers per group (small, readable)
    for i, x in enumerate(groups, start=1):
        if x.size == 0:
            continue
        mu, lo, hi = bootstrap_mean_ci(x, n_boot=BOOT_N, ci=CI, seed=RNG_SEED)
        ax.plot([i, i], [lo, hi], linewidth=2.0)          # CI as vertical segment
        ax.plot(i, mu, marker="o", markersize=4.5)        # mean dot

    ax.set_title(f"{title} — {SPLIT} (GO={GO_ASPECT})")
    ax.set_ylabel(metric)
    ax.set_xticks(pos)
    ax.set_xticklabels(xticks, rotation=25, ha="right")

    xlim = opts.get("xlim", None)
    if xlim is not None:
        lo_, hi_ = xlim
        if hi_ is None:
            hi_ = np.nanpercentile(df_all[metric].astype(float), 99.5)
            if not np.isfinite(hi_):
                hi_ = None
        ax.set_ylim(lo_, hi_) if metric in ("hit_any_shared_go", "shared_go_frac_neighbors", "go_entropy_neighborhood") else None
        # (for these 0..1 metrics, ylim is more useful than xlim)
        if metric in ("hit_any_shared_go", "shared_go_frac_neighbors", "go_entropy_neighborhood"):
            ax.set_ylim(lo_, hi_)

    fig.tight_layout()
    #plt.show()
    #savefig_dual(fig, OUT_DIR / f"violinbox_{metric}_{SPLIT}")

print(f"[OK] Wrote violin/box plots to: {OUT_DIR}")


In [ ]:
df_all.groupby(["method", "tag"]).agg({
    "shared_go_frac_neighbors": ["mean", "std"],
    "hit_any_shared_go": ["mean", "std"],
    "neglog10_best_pval": ["mean", "std"],
    "go_entropy_neighborhood": ["mean", "std"],
}).round(4)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------- settings ----------
TAGS = ["knn_true", "control_shuffled_embeddings", "control_random_neighbors"]
TAG_PRETTY = {
    "knn_true": "true kNN",
    "control_shuffled_embeddings": "shuffled emb",
    "control_random_neighbors": "random neigh",
}
BOOT_N = 2000
CI = 0.95
SEED = 0

MAIN_METRICS = [
    ("shared_go_frac_neighbors", "Neighborhood purity (shared GO fraction)", dict(ylim=(0, 1))),
    ("neglog10_best_pval", "Neighborhood enrichment (-log10 best Fisher p)", dict()),  # ylim decided later (clipped)
]

def bootstrap_mean_ci(x, n_boot=2000, ci=0.95, seed=0):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    means = []
    for _ in range(int(n_boot)):
        samp = rng.choice(x, size=x.size, replace=True)
        means.append(np.mean(samp))
    means = np.asarray(means)
    a = (1-ci)/2
    return float(np.mean(x)), float(np.quantile(means, a)), float(np.quantile(means, 1-a))

# Optional: robust clipping for enrichment so bulk is visible
if "neglog10_best_pval" in df_all.columns:
    clip_hi = float(np.nanpercentile(df_all["neglog10_best_pval"].astype(float), 99.5))
    if np.isfinite(clip_hi) and clip_hi > 0:
        df_all = df_all.copy()
        df_all["neglog10_best_pval_clip"] = np.minimum(df_all["neglog10_best_pval"].astype(float), clip_hi)
        print(f"[INFO] Clipping neglog10_best_pval at 99.5th percentile: {clip_hi:.2f}")
        MAIN_METRICS = [
            ("shared_go_frac_neighbors", "Neighborhood purity (shared GO fraction)", dict(ylim=(0, 1))),
            ("neglog10_best_pval_clip", f"Neighborhood enrichment (-log10 best Fisher p), clipped @ {clip_hi:.1f}", dict()),
        ]

# ---------- Figure 1: overlaid distributions (best for reviewers) ----------
fig = plt.figure(figsize=(6, 2 * len(MAIN_METRICS)))
for r, (mcol, title, opts) in enumerate(MAIN_METRICS, start=1):
    for c, method in enumerate(METHODS, start=1):
        ax = fig.add_subplot(len(MAIN_METRICS), len(METHODS), (r-1)*len(METHODS) + c)

        # overlay hist densities per tag
        for tag in TAGS:
            x = df_all[(df_all["method"] == method) & (df_all["tag"] == tag)][mcol].to_numpy(dtype=float)
            x = x[np.isfinite(x)]
            if x.size == 0:
                continue

            ax.hist(x, bins=45, density=True, alpha=0.35, label=TAG_PRETTY.get(tag, tag))

            mu, lo, hi = bootstrap_mean_ci(x, n_boot=BOOT_N, ci=CI, seed=SEED)
            ax.axvline(mu, linestyle="--", linewidth=1.5)
            ax.axvspan(lo, hi, alpha=0.12)

        # annotate effect: Δ(true - shuffled)
        xt = df_all[(df_all["method"] == method) & (df_all["tag"] == "knn_true")][mcol].to_numpy(dtype=float)
        xs = df_all[(df_all["method"] == method) & (df_all["tag"] == "control_shuffled_embeddings")][mcol].to_numpy(dtype=float)
        xt = xt[np.isfinite(xt)]
        xs = xs[np.isfinite(xs)]
        if xt.size and xs.size:
            d = np.mean(xt) - np.mean(xs)
            ax.text(0.99, 0.95, f"Δ(true−shuf)={d:.3f}", ha="right", va="top", transform=ax.transAxes)

        ax.set_title(f"{method}: {title}")
        ax.set_ylabel("Density")
        ax.set_xlabel(mcol)

        if "ylim" in opts:
            ax.set_ylim(*opts["ylim"])

        ax.legend(loc="best", frameon=True)

fig.tight_layout()
plt.show()

# ---------- Figure 2: grouped violins (compact, less clutter) ----------
for mcol, title, opts in MAIN_METRICS:
    fig = plt.figure(figsize=(6, 2))
    ax = fig.add_subplot(111)

    # positions: group by method, within each group: tag
    positions = []
    data = []
    labels = []
    pos = 1
    gap = 1.2

    for method in METHODS:
        for tag in TAGS:
            x = df_all[(df_all["method"] == method) & (df_all["tag"] == tag)][mcol].to_numpy(dtype=float)
            x = x[np.isfinite(x)]
            data.append(x)
            positions.append(pos)
            labels.append(f"{method}\n{TAG_PRETTY.get(tag, tag)}")
            pos += 1
        pos += gap

    ax.violinplot(data, positions=positions, showmeans=False, showmedians=True, showextrema=False)
    ax.boxplot(data, positions=positions, widths=0.25, showfliers=False)

    # mean + CI markers
    for p, x in zip(positions, data):
        if x.size == 0:
            continue
        mu, lo, hi = bootstrap_mean_ci(x, n_boot=BOOT_N, ci=CI, seed=SEED)
        ax.plot([p, p], [lo, hi], linewidth=2.0)
        ax.plot(p, mu, marker="o", markersize=4.5)

    ax.set_title(title)
    ax.set_xticks(positions)
    ax.set_xticklabels(labels, rotation=20, ha="right")
    ax.set_ylabel(mcol)

    if "ylim" in opts:
        ax.set_ylim(*opts["ylim"])

    fig.tight_layout()
    plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# -----------------------
# REQUIRED: df_all
# columns expected:
#   method (str): one of {"puffin","mincut","esm-kmeans"} OR already pretty names
#   tag (str): {"knn_true","control_shuffled_embeddings", ...}
#   shared_go_frac_neighbors
#   neglog10_best_pval or neglog10_best_pval_clip
# -----------------------

# Pretty names + fixed colors
METHOD_PRETTY = {
    "puffin": "PUFFIN (func-aware)",
    "mincut": "MinCut (struct-only)",
    "esm-kmeans": "ESM-KMeans (unsup)",
    # if your df already stores pretty names, it will pass through unchanged
}
COLOR = {
    "PUFFIN (func-aware)":  "C0",
    "MinCut (struct-only)": "C1",
    "ESM-KMeans (unsup)":      "C2",
}

# Keep only the two conditions you want
KEEP_TAGS = ["knn_true", "control_shuffled_embeddings"]
KEEP_TAGS = ["knn_true"]
TAG_PRETTY = {
    "knn_true": "true kNN",
    "control_shuffled_embeddings": "shuffled emb",
}

BOOT_N = 2000
CI = 0.95
SEED = 0

def bootstrap_mean_ci(x, n_boot=2000, ci=0.95, seed=0):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    means = []
    for _ in range(int(n_boot)):
        samp = rng.choice(x, size=x.size, replace=True)
        means.append(np.mean(samp))
    means = np.asarray(means)
    a = (1-ci)/2
    return float(np.mean(x)), float(np.quantile(means, a)), float(np.quantile(means, 1-a))

def _get_method_list(df):
    ms = []
    for m in df["method"].astype(str).unique():
        mp = METHOD_PRETTY.get(m, m)
        if mp in COLOR:
            ms.append(mp)
    # keep stable PUFFIN/MinCut/ESM-KMeans order if present
    order = ["PUFFIN (func-aware)", "ESM-KMeans (unsup)", "MinCut (struct-only)"]
    ms = [m for m in order if m in ms] + [m for m in ms if m not in order]
    return ms

# normalize method names in df
df = df_all.copy()
df["method_pretty"] = df["method"].astype(str).map(lambda x: METHOD_PRETTY.get(x, x))
df = df[df["tag"].isin(KEEP_TAGS)].copy()

# choose enrichment column
if "neglog10_best_pval_clip" in df.columns:
    ENR_COL = "neglog10_best_pval_clip"
    ENR_TITLE = "Neighborhood enrichment (-log10 best Fisher p), clipped"
else:
    ENR_COL = "neglog10_best_pval"
    ENR_TITLE = "Neighborhood enrichment (-log10 best Fisher p)"

# -----------------------
# FIGURE 1: Overlay histograms per method (2 rows: purity + enrichment; columns=methods)
# -----------------------
methods = _get_method_list(df)

fig = plt.figure(figsize=(4.6 * len(methods), 7.2))

# Row 1: purity
for ci, m in enumerate(methods, start=1):
    ax = fig.add_subplot(2, len(methods), ci)

    col = COLOR[m]
    for tag in KEEP_TAGS:
        x = df[(df["method_pretty"] == m) & (df["tag"] == tag)]["shared_go_frac_neighbors"].to_numpy(dtype=float)
        x = x[np.isfinite(x)]
        if x.size == 0:
            continue

        if tag == "knn_true":
            ax.hist(x, bins=45, density=True, alpha=0.35, color=col, label=TAG_PRETTY[tag])
        else:
            ax.hist(x, bins=45, density=True, alpha=0.18, color=col, edgecolor=col, linewidth=1.0, label=TAG_PRETTY[tag])

        mu, lo, hi = bootstrap_mean_ci(x, n_boot=BOOT_N, ci=CI, seed=SEED)
        ax.axvline(mu, linestyle="--", linewidth=1.6, color=col, alpha=0.9)

    # Δ annotation (true - shuffled)
    xt = df[(df["method_pretty"] == m) & (df["tag"] == "knn_true")]["shared_go_frac_neighbors"].to_numpy(dtype=float)
    xs = df[(df["method_pretty"] == m) & (df["tag"] == "control_shuffled_embeddings")]["shared_go_frac_neighbors"].to_numpy(dtype=float)
    xt = xt[np.isfinite(xt)]
    xs = xs[np.isfinite(xs)]
    if xt.size and xs.size:
        d = float(np.mean(xt) - np.mean(xs))
        ax.text(0.98, 0.95, f"Δ(true−shuf)={d:.3f}", ha="right", va="top", transform=ax.transAxes)

    ax.set_title(f"{m}\nNeighborhood purity")
    ax.set_xlabel("shared_go_frac_neighbors")
    ax.set_ylabel("Density")
    ax.set_xlim(0, 1)
    ax.legend(loc="upper right", frameon=True)

# Row 2: enrichment
for ci, m in enumerate(methods, start=1):
    ax = fig.add_subplot(2, len(methods), len(methods) + ci)

    col = COLOR[m]
    for tag in KEEP_TAGS:
        x = df[(df["method_pretty"] == m) & (df["tag"] == tag)][ENR_COL].to_numpy(dtype=float)
        x = x[np.isfinite(x)]
        if x.size == 0:
            continue

        if tag == "knn_true":
            ax.hist(x, bins=60, density=True, alpha=0.35, color=col, label=TAG_PRETTY[tag])
        else:
            ax.hist(x, bins=60, density=True, alpha=0.18, color=col, edgecolor=col, linewidth=1.0, label=TAG_PRETTY[tag])

        mu, lo, hi = bootstrap_mean_ci(x, n_boot=BOOT_N, ci=CI, seed=SEED)
        ax.axvline(mu, linestyle="--", linewidth=1.6, color=col, alpha=0.9)

    xt = df[(df["method_pretty"] == m) & (df["tag"] == "knn_true")][ENR_COL].to_numpy(dtype=float)
    xs = df[(df["method_pretty"] == m) & (df["tag"] == "control_shuffled_embeddings")][ENR_COL].to_numpy(dtype=float)
    xt = xt[np.isfinite(xt)]
    xs = xs[np.isfinite(xs)]
    if xt.size and xs.size:
        d = float(np.mean(xt) - np.mean(xs))
        ax.text(0.98, 0.95, f"Δ(true−shuf)={d:.3f}", ha="right", va="top", transform=ax.transAxes)

    ax.set_title(f"{m}\n{ENR_TITLE}")
    ax.set_xlabel(ENR_COL)
    ax.set_ylabel("Density")
    ax.legend(loc="upper right", frameon=True)

fig.tight_layout()
plt.show()

# -----------------------
# FIGURE 2: Compact violin+box (grouped by method; two violins per method)
# -----------------------
def grouped_violin_box(metric_col, title, ylim=None):
    fig = plt.figure(figsize=(12.5, 4.6))
    ax = fig.add_subplot(111)

    positions = []
    data = []
    colors = []
    labels = []
    pos = 1
    gap = 1.1

    for m in methods:
        col = COLOR[m]
        for tag in KEEP_TAGS:
            x = df[(df["method_pretty"] == m) & (df["tag"] == tag)][metric_col].to_numpy(dtype=float)
            x = x[np.isfinite(x)]
            data.append(x)
            positions.append(pos)
            colors.append(col)
            labels.append(f"{m}\n{TAG_PRETTY[tag]}")
            pos += 1
        pos += gap

    vp = ax.violinplot(data, positions=positions, showmeans=False, showmedians=True, showextrema=False)
    for i, body in enumerate(vp["bodies"]):
        body.set_facecolor(colors[i])
        body.set_edgecolor(colors[i])
        body.set_alpha(0.22 if (i % 2 == 1) else 0.35)  # shuffled lighter
        body.set_linewidth(1.0)

    bp = ax.boxplot(data, positions=positions, widths=0.25, showfliers=False, patch_artist=False)

    # means as dots
    for p, x, col in zip(positions, data, colors):
        if x.size == 0:
            continue
        mu, lo, hi = bootstrap_mean_ci(x, n_boot=BOOT_N, ci=CI, seed=SEED)
        ax.plot([p, p], [lo, hi], linewidth=2.0, color=col, alpha=0.9)
        ax.plot(p, mu, marker="o", markersize=4.8, color=col)

        # horizontal mean bar
        ax.text(
            p,
            hi + 0.03,
            f"{mu:.2f} [{lo:.2f}, {hi:.2f}]",
            ha="center",
            va="bottom",
            fontsize=8,
            color=col,
        )



    ax.set_title(title)
    ax.set_xticks(positions)
    ax.set_xticklabels(labels, rotation=18, ha="right")
    ax.set_ylabel(metric_col)
    if ylim is not None:
        ax.set_ylim(*ylim)
    fig.tight_layout()
    plt.show()

grouped_violin_box(
    "shared_go_frac_neighbors",
    "Neighborhood purity (shared GO fraction): true kNN vs shuffled embeddings",
    ylim=(0,1),
)

grouped_violin_box(
    ENR_COL,
    f"{ENR_TITLE}: true kNN vs shuffled embeddings",
    ylim=None,
)


In [ ]:


KEEP_METHODS = ["puffin", "esm-kmeans", "mincut"]
METHOD_LABEL = {
    "puffin": "PUFFIN (func-aware)",
    "esm-kmeans": "ESM-KMeans (unsup)",
    "mincut": "MinCut (struct-only)",
}
def _prep_no_controls(per_query: pd.DataFrame) -> pd.DataFrame:
    df = per_query.copy()
    # keep only real methods + true kNN
    if "method" in df.columns:
        df = df[df["method"].isin(KEEP_METHODS)]
    else:
        raise ValueError("per_query must include a 'method' column (puffin/esm-kmeans/mincut).")
    if "tag" in df.columns:
        df = df[df["tag"].isin(KEEP_TAGS)]
    else:
        raise ValueError("per_query must include a 'tag' column (e.g., knn_true).")

    df["method_label"] = df["method"].map(METHOD_LABEL)
    # stable plotting order
    df["method_label"] = pd.Categorical(
        df["method_label"],
        categories=["PUFFIN (func-aware)", "ESM-KMeans (unsup)", "MinCut (struct-only)"],
        ordered=True,
    )
    return df

def _annotate_stats(ax, x, y, s, dy=0.02):
    ax.text(
        x, y,
        s,
        ha="center", va="bottom",
        fontsize=10,
    )

def grouped_violin_box_no_controls(
    per_query: pd.DataFrame,
    ycol: str,
    title: str,
    *,
    ylim=None,
    ylabel=None,
    show_numbers=True,      # adds median/mean + dashed mean line
    mean_as_dot=True,
    ax=None,
):
    if ax is None:
        fig, ax = plt.subplots(1, 1, figsize=(10.5, 4.6))
    else:
        fig = ax.figure

    df = _prep_no_controls(per_query)
    
    methods = [c for c in df["method_label"].cat.categories if c in df["method_label"].unique()]
    data = []
    for m in methods:
        v = df.loc[df["method_label"] == m, ycol].to_numpy(dtype=float)
        v = v[np.isfinite(v)]
        data.append(v)

    # violin (colored per method)
    parts = ax.violinplot(data, showmeans=False, showmedians=False, showextrema=False)
    for body, m in zip(parts["bodies"], methods):
        body.set_alpha(0.25)
        body.set_facecolor(COLOR[m])

    # box (same positions)
    bp = ax.boxplot(
        data,
        widths=0.18,
        vert=True,
        showfliers=False,
        patch_artist=True,
    )
    for patch, m in zip(bp["boxes"], methods):
        patch.set_facecolor((1, 1, 1, 0))  # transparent
        patch.set_edgecolor("black")
        patch.set_linewidth(1.2)

    # median line styling
    for med in bp["medians"]:
        med.set_color("black")
        med.set_linewidth(1.5)

    # mean dashed line + mean dot + numbers
    for i, (m, v) in enumerate(zip(methods, data), start=1):
        if v.size == 0:
            continue
        mu = float(np.mean(v))
        med = float(np.median(v))
        ax.hlines(mu, i - 0.22, i + 0.22, linestyles="--", linewidth=2.0)
        if mean_as_dot:
            ax.scatter([i], [mu], s=45, zorder=3)
        if show_numbers:
            _annotate_stats(
                ax, i, (ylim[1] if ylim else ax.get_ylim()[1]),
                f"mean={mu:.3f}\nmed={med:.3f}",
                dy=0.02,
            )

    ax.set_title(title)
    ax.set_xticks(np.arange(1, len(methods) + 1))
    ax.set_xticklabels(methods, rotation=20, ha="right")
    ax.set_ylabel(ylabel or ycol)
    if ylim is not None:
        ax.set_ylim(*ylim)

    # keep some headroom if we annotate at the top
    if show_numbers:
        lo, hi = ax.get_ylim()
        ax.set_ylim(lo, hi + (hi - lo) * 0.12)

    fig.tight_layout()
    return fig, ax

fig, axs = plt.subplots(2, 1, figsize=(10.5, 9.6))


_, ax1 = grouped_violin_box_no_controls(
    df_all,
    "shared_go_frac_neighbors",
    "Neighborhood purity (shared GO fraction)",
    ylim=(0, 1),
    ylabel="shared_go_frac_neighbors",
    show_numbers=True,
    ax=axs[0],
)
_panel_label(ax1, "A")
# plt.show()



ENR_COL = "neglog10_best_pval_clip" if "neglog10_best_pval_clip" in df_all.columns else "neglog10_best_pval"
_, ax2 = grouped_violin_box_no_controls(
    df_all,
    ENR_COL,
    "Neighborhood enrichment (-log10 best Fisher p)",
    ylim=None,
    ylabel=ENR_COL,
    show_numbers=True,
    ax=axs[1],
)
_panel_label(ax2, "B")
# plt.show()


In [ ]:
!pip install brokenaxes
from brokenaxes import brokenaxes
import numpy as np
import matplotlib.pyplot as plt

def grouped_violin_box_no_controls_broken(
    per_query: pd.DataFrame,
    ycol: str,
    title: str,
    *,
    ylims=((0, 25), (90, 100)),   # <-- set your low/high windows here
    ylabel=None,
    show_numbers=True,
    mean_as_dot=True,
    fig=None,
    subplot_spec=None,           # <-- GridSpec slot
    hspace=0.05,
):
    if fig is None or subplot_spec is None:
        raise ValueError("Provide fig=... and subplot_spec=... when using brokenaxes.")

    df = _prep_no_controls(per_query)
    methods = [c for c in df["method_label"].cat.categories if c in df["method_label"].unique()]

    data = []
    for m in methods:
        v = df.loc[df["method_label"] == m, ycol].to_numpy(dtype=float)
        v = v[np.isfinite(v)]
        data.append(v)

    bax = brokenaxes(
        ylims=ylims,
        hspace=hspace,
        subplot_spec=subplot_spec,
        fig=fig,
        despine=False,
    )

    # violin + box on each internal axis
    # (brokenaxes proxies most methods to all segments internally)
    parts = bax.violinplot(data, showmeans=False, showmedians=False, showextrema=False)

    # brokenaxes may return:
    #  - dict (matplotlib style)
    #  - list of dicts (one per internal axis)
    #  - list of artists
    bodies = []

    if isinstance(parts, dict):
        bodies = parts.get("bodies", [])
    elif isinstance(parts, (list, tuple)):
        # list of dicts
        if len(parts) > 0 and isinstance(parts[0], dict):
            for d in parts:
                bodies.extend(d.get("bodies", []))
        else:
            # sometimes it returns a flat list of PolyCollections (already bodies)
            bodies = list(parts)

    # Color violins: we expect 1 violin body per method per segment-axis.
    # So bodies are typically repeated in blocks of len(methods).
    n_methods = len(methods)
    for bi, body in enumerate(bodies):
        m = methods[bi % n_methods]  # cycle across methods
        body.set_alpha(0.25)
        body.set_facecolor(COLOR[m])

    bp = bax.boxplot(
        data,
        widths=0.18,
        vert=True,
        showfliers=True,
        patch_artist=True,
    )

    # brokenaxes may return:
    #  - dict
    #  - list of dicts (one per y-segment)
    boxes = []
    medians = []

    if isinstance(bp, dict):
        boxes = bp.get("boxes", [])
        medians = bp.get("medians", [])
    elif isinstance(bp, (list, tuple)):
        for part in bp:
            if isinstance(part, dict):
                boxes.extend(part.get("boxes", []))
                medians.extend(part.get("medians", []))

    # Style boxes (cycle methods across segments)
    n_methods = len(methods)
    for i, patch in enumerate(boxes):
        m = methods[i % n_methods]
        patch.set_facecolor((1, 1, 1, 0))  # transparent
        patch.set_edgecolor("black")
        patch.set_linewidth(1.2)

    # Style medians
    for med in medians:
        med.set_color("black")
        med.set_linewidth(1.5)

    # mean dashed + dot + numbers (place numbers in the TOP broken segment)
    # We annotate on the *top* internal axes object so it appears above everything.
    top_ax = bax.axs[0]  # top segment axis
    y_top_lo, y_top_hi = top_ax.get_ylim()

    for i, (m, v) in enumerate(zip(methods, data), start=1):
        if v.size == 0:
            continue
        mu = float(np.mean(v))
        med = float(np.median(v))

        bax.hlines(mu, i - 0.22, i + 0.22, linestyles="--", linewidth=2.0)
        if mean_as_dot:
            bax.scatter([i], [mu], s=45, zorder=3)

        if show_numbers:
            top_ax.text(
                i, y_top_hi,
                f"mean={mu:.3f}\nmed={med:.3f}",
                ha="center", va="bottom", fontsize=10
            )

    bax.set_title(title)
    bax.set_xticks(np.arange(1, len(methods) + 1))
    bax.set_xticklabels(methods, rotation=20, ha="right")
    bax.set_ylabel(ylabel or ycol)

    # add a bit of headroom on the TOP segment only
    if show_numbers:
        top_ax.set_ylim(y_top_lo, y_top_hi + (y_top_hi - y_top_lo) * 0.12)

    return bax

KEEP_TAGS = ["knn_true"]   # make sure this exists
# COLOR must already exist with your method-label keys:
# COLOR = {
#   "PUFFIN (func-aware)": "C0",
#   "ESM-KMeans (unsup)": "C2",
#   "MinCut (struct-only)": "C1",
# }

fig = plt.figure(figsize=(10.5, 9.6))
gs = fig.add_gridspec(2, 1, height_ratios=[1, 1.15], hspace=0.28)

# Panel A: normal axes
ax1 = fig.add_subplot(gs[0, 0])
_, ax1 = grouped_violin_box_no_controls(
    df_all,
    "shared_go_frac_neighbors",
    "Neighborhood purity (shared GO fraction)",
    ylim=(0, 1),
    ylabel="shared_go_frac_neighbors",
    show_numbers=True,
    ax=ax1,
)
_panel_label(ax1, "A")

# Panel B: broken y-axis (enrichment)
ENR_COL = "neglog10_best_pval_clip" if "neglog10_best_pval_clip" in df_all.columns else "neglog10_best_pval"

bax2 = grouped_violin_box_no_controls_broken(
    df_all,
    ENR_COL,
    "Neighborhood enrichment (-log10 best Fisher p)",
    ylims=((-0.5, 15), (30, 110)),   # <-- tune these two windows for your data
    ylabel=ENR_COL,
    show_numbers=True,
    fig=fig,
    subplot_spec=gs[1, 0],
    hspace=0.05,
)

# panel label for brokenaxes: label on the top segment axis
_panel_label(bax2.axs[0], "B")

plt.show()


In [ ]:
df_all.groupby(["method", "tag"]).agg({
    "neglog10_best_pval": ["mean", "std", "median", "min", "max"],
}).round(4)

In [ ]:
import numpy as np

def add_log2_best_odds(df, col="best_odds_approx", out="log2_best_odds", clip=128.0):
    x = df[col].to_numpy(dtype=float)
    x = np.where(np.isfinite(x), x, np.nan)
    # avoid inf/0
    # x = np.clip(x, 1.0/clip, clip)
    df[out] = np.log2(x)
    return df

df_all = add_log2_best_odds(df_all, clip=None)

fig, ax = grouped_violin_box_no_controls(
    df_all,
    "log2_best_odds",
    "Neighborhood enrichment (log2 odds ratio)",
    # ylim=(-7, 7),  # since clip=128 => +/-7
    ylabel="log2(best odds ratio)",
    show_numbers=True,
)
plt.show()


In [ ]:

fig, axs = plt.subplots(2, 1, figsize=(10.5, 9.6))


_, ax1 = grouped_violin_box_no_controls(
    df_all,
    "shared_go_frac_neighbors",
    "Neighborhood purity (shared GO fraction)",
    ylim=(0, 1),
    ylabel="shared_go_frac_neighbors",
    show_numbers=True,
    ax=axs[0],
)
_panel_label(ax1, "A")
# plt.show()



_, ax2 = grouped_violin_box_no_controls(
    df_all,
    "log2_best_odds",
    "Neighborhood enrichment (log2 odds ratio)",
    # ylim=(-7, 7),  # since clip=128 => +/-7
    ylabel="log2(best odds ratio)",
    show_numbers=True,
    ax=axs[1],
)
_panel_label(ax2, "B")
# plt.show()


In [ ]:
df_all.groupby(["method", "tag"])["neglog10_best_pval"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).round(4)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

COLOR = {
    "PUFFIN (func-aware)":  "C0",
    "MinCut (struct-only)": "C1",
    "ESM-KMeans (unsup)":      "C2",
}
METHOD_LABEL = {
    "puffin": "PUFFIN (func-aware)",
    "mincut": "MinCut (struct-only)",
    "esm-kmeans": "ESM-KMeans (unsup)",
}

MARKER = {
    "PUFFIN (func-aware)": "o",
    "ESM-KMeans (unsup)": "^",
    "MinCut (struct-only)": "s",
}

def scatter_alignment_vs_diversity_no_controls(
    per_query: pd.DataFrame,
    *,
    xcol="neff_neighbor_proteins",
    ycol="shared_go_frac_neighbors",
    title="Function alignment vs diversity",
    xlim=None,
    ylim=None,
    max_points_per_method=2500,   # subsample for readability
    seed=0,
):
    df = per_query.copy()

    # keep only real methods + true kNN
    df = df[df["method"].isin(METHOD_LABEL.keys())]
    df = df[df["tag"] == "knn_true"].copy()
    df["method_label"] = df["method"].map(METHOD_LABEL)

    # finite only
    df = df[np.isfinite(df[xcol].to_numpy(dtype=float)) & np.isfinite(df[ycol].to_numpy(dtype=float))]

    rng = np.random.default_rng(seed)

    fig, ax = plt.subplots(figsize=(8.8, 6.2))

    order = ["PUFFIN (func-aware)", "ESM-KMeans (unsup)", "MinCut (struct-only)"]
    for m in order:
        d = df[df["method_label"] == m]
        if len(d) == 0:
            continue

        # subsample
        if len(d) > max_points_per_method:
            d = d.sample(n=max_points_per_method, random_state=seed)

        x = d[xcol].to_numpy(dtype=float)
        y = d[ycol].to_numpy(dtype=float)

        ax.scatter(
            x, y,
            s=14,
            alpha=0.28,
            color=COLOR[m],
            marker=MARKER[m],
            edgecolor="black",
            linewidth=0.25,
            rasterized=True,
        )

        # add method centroid (mean) for quick reading
        ax.scatter(
            [float(np.mean(x))], [float(np.mean(y))],
            s=90,
            marker=MARKER[m],
            color=COLOR[m],
            edgecolor="black",
            linewidth=0.8,
            zorder=5,
        )

    ax.set_title(title)
    ax.set_xlabel("neff_neighbor_proteins (Simpson inverse)")
    ax.set_ylabel("shared_go_frac_neighbors")
    max_neff = df[xcol].to_numpy(dtype=float).max()
    ax.set_ylim(-0.02, 1.02)
    ax.set_xlim(-1, max_neff + 1)   # e.g. 51 if k=50

    if xlim is not None:
        ax.set_xlim(*xlim)
    if ylim is not None:
        ax.set_ylim(*ylim)
    ax.annotate("Higher functional alignment for similar diversity",
            xy=(28, 0.75), xytext=(10, 0.55),
            arrowprops=dict(arrowstyle="->", lw=1.2))
    ax.legend(loc="upper right", frameon=True)
    
    fig.tight_layout()
    return fig, ax

fig, ax = scatter_alignment_vs_diversity_no_controls(df_all)

legend_elements = []
for m in METHOD_LABEL.keys():
    legend_elements.append(
        Line2D([0], [0],
               marker=MARKER[METHOD_LABEL[m]], color='w',
               label=METHOD_LABEL[m],
               markerfacecolor=COLOR[METHOD_LABEL[m]],
               markeredgecolor='black',
               markersize=8)
    )

ax.legend(
    handles=legend_elements,
    loc="upper right",
    frameon=True,
    title="Model",
)
ax.set_xlabel("Effective neighbor proteins)")
ax.set_ylabel("Shared GO fraction neighbors")
fig.tight_layout()
plt.show()


In [ ]:
# Notebook cell: plotting "Does functional supervision reshape functional unit discovery?"
# Metrics:
# 1) Prototype functional purity: GO entropy per prototype
# 2) Prototype reuse: #proteins per prototype
# 3) Boundary shift vs structure-only MinCut: residue-label change + boundary change stats
#
# Assumes you have (per method):
#   - proto enrichment CSV (optional; used only for labeling top terms)
#       columns: proto, go_term, ..., proto_proteins, total_proteins, pval, odds_ratio_approx, qval
#   - residue assignment CSV:
#       columns: db_id, n_residues, residue_ids, cluster_ids, K
# and a GO annotation TSV like nrPDB-GO_annot.tsv (CAFA-style: PDB, MF, BP, CC)

import re
import math
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt



GO_TSV = Path("../data/GeneOntology/nrPDB-GO_annot.tsv")   
GO_ASPECT = "MF"                             # MF/BP/CC

K = 512

METHODS = {
    # method_name: dict(assignments=..., proto_enrich=optional)
    "puffin_v4": {
        "label": "PUFFIN (func-aware) v4",
        "assignments": Path("../ismb26/segments/puffin_K64_v4/train/train_residue_assignments.csv"),  
        "proto_enrich": Path(f"../ismb26/prototypes/puffin_K64_v4/K{K}/eval/go_enrichment_train_all.csv"),        
    },
    "esm-kmeans": {
        "label": "ESM-KMeans (unsup)",
        "assignments": Path("../ismb26/segments/esm-kmeans/train/train_residue_assignments.csv"),
        "proto_enrich": Path(f"../ismb26/prototypes/esm-kmeans/K{K}/eval/go_enrichment_train_all.csv"),     
    },
    "mincut": {
        "label": "MinCut (struct-only)",
        "assignments": Path("../ismb26/segments/mincut_K64/train/train_residue_assignments.csv"),
        "proto_enrich": Path(f"../ismb26/prototypes/mincut_K64/K{K}/eval/go_enrichment_train_all.csv"),       
    },
}
STRUCT_BASELINE = "mincut"  # boundary shift baseline


# -----------------------
# IO helpers
# -----------------------
def load_go_tsv(path: Path) -> pd.DataFrame:
    # tries common skiprows like your pipeline
    for skip in (12, 14, 13):
        try:
            df = pd.read_csv(path, sep="\t", skiprows=skip)
            if df.shape[1] >= 4:
                df = df.iloc[:, :4].copy()
                df.columns = ["PDB", "MF", "BP", "CC"]
                df["PDB"] = df["PDB"].astype(str)
                return df.set_index("PDB")
        except Exception:
            pass
    raise ValueError(f"Could not parse GO TSV: {path}")

def safe_go_list(val) -> list[str]:
    if val is None:
        return []
    if isinstance(val, float) and np.isnan(val):
        return []
    s = str(val).strip()
    if not s or s.lower() == "nan":
        return []
    return [t for t in s.split(",") if t]

def split_id(pdb_id: str):
    s = str(pdb_id)
    if "_" in s:
        left, _ = s.split("_", 1)
    else:
        left = s
    if "-" in left:
        pdb, chain = left.split("-", 1)
        return pdb.upper(), chain
    return left.upper(), "ALL"

def legacy_id_from_new(pdb_id: str) -> str:
    pdb, chain = split_id(pdb_id)
    return f"{pdb}-{chain}"

def parse_int_list_csv(s: str) -> np.ndarray:
    # handles "0,1,2" optionally inside quotes
    s = str(s).strip().strip('"').strip()
    if s == "":
        return np.array([], dtype=int)
    return np.fromiter((int(x) for x in s.split(",") if x != ""), dtype=int)

def load_assignments_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    need = {"pdb_id", "n_residues", "residue_ids", "cluster_ids"}
    miss = need - set(df.columns)
    if miss:
        raise ValueError(f"{path} missing columns: {sorted(miss)}")
    df = df.copy()
    df["protein_key"] = df["pdb_id"].astype(str).apply(legacy_id_from_new)
    df["n_residues"] = pd.to_numeric(df["n_residues"], errors="coerce").astype(int)
    return df

def assignment_to_labels_on_residue_ids(row: pd.Series):
    rid = parse_int_list_csv(row["residue_ids"])
    cid = parse_int_list_csv(row["cluster_ids"])

    if rid.size == 0 or cid.size == 0:
        return np.array([], dtype=int), np.array([], dtype=int)

    m = min(rid.size, cid.size)
    rid = rid[:m].astype(int)
    cid = cid[:m].astype(int)

    # If residue_ids can repeat, keep the last assignment (or change to majority vote)
    # We'll do "last wins" via dict overwrite:
    assign = {}
    for r, c in zip(rid, cid):
        assign[int(r)] = int(c)

    rid_sorted = np.array(sorted(assign.keys()), dtype=int)
    labels = np.array([assign[int(r)] for r in rid_sorted], dtype=int)
    return labels, rid_sorted

import numpy as np
import pandas as pd
from pathlib import Path

def load_assignment_split(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    need = {"protein_key", "segment_k", "proto"}
    miss = need - set(df.columns)
    if miss:
        raise ValueError(f"{path} missing columns: {sorted(miss)}")

    df = df.copy()
    df["protein_key"] = df["protein_key"].astype(str)
    df["segment_k"]   = pd.to_numeric(df["segment_k"], errors="coerce")
    df["proto"]       = pd.to_numeric(df["proto"], errors="coerce")
    df = df.dropna(subset=["protein_key", "segment_k", "proto"])
    df["segment_k"] = df["segment_k"].astype(int)
    df["proto"]     = df["proto"].astype(int)
    return df

def build_protein_segment_to_proto_map(assign_df: pd.DataFrame) -> dict[str, dict[int, int]]:
    """
    Returns:
      per_protein_map[protein_key][segment_k] = proto
    """
    per_protein = {}
    for pk, g in assign_df.groupby("protein_key"):
        # if duplicates exist for same (pk, segment_k), keep the most common proto (or last)
        # Here: majority vote
        mp = {}
        for seg_k, gg in g.groupby("segment_k"):
            proto = int(gg["proto"].mode().iloc[0])
            mp[int(seg_k)] = proto
        per_protein[str(pk)] = mp
    return per_protein

def build_method_proto_dense_from_assignment_split(
    method: str,
    assignment_split_path: Path,
    *,
    default_proto: int = -1,
):
    """
    Builds:
      method_proto_dense[method][protein] = y_proto (per-residue prototype ids)
    using:
      method_dense[method] = y_seg (per-residue segment_k ids)
      method_resids[method] = residue ids
      assignment_split.csv = mapping (protein_key, segment_k) -> proto
    """
    assign_df = load_assignment_split(assignment_split_path)
    per_protein_map = build_protein_segment_to_proto_map(assign_df)

    d_lab, d_rid = {}, {}
    missing_map_proteins = 0
    missing_seg_assigns = 0

    for p in common_proteins:
        r = method_resids[method][p]
        y_seg = method_dense[method][p]  # segment_k per residue
        if r.size == 0 or y_seg.size == 0:
            d_lab[p] = np.array([], dtype=int)
            d_rid[p] = np.array([], dtype=int)
            continue

        # IMPORTANT: keys must match.
        # Your method_dense proteins are keyed by legacy_id style like "3ONG-B".
        # Your assignment_split has protein_key like "3ONG-B" (good).
        pk = str(p)

        seg2proto = per_protein_map.get(pk, None)
        if seg2proto is None:
            missing_map_proteins += 1
            y_proto = np.full_like(y_seg, fill_value=default_proto, dtype=int)
        else:
            y_proto = np.full_like(y_seg, fill_value=default_proto, dtype=int)
            # map segment_k -> proto per residue
            for i, seg_k in enumerate(y_seg.tolist()):
                if seg_k >= 0:
                    proto = seg2proto.get(int(seg_k), default_proto)
                    if proto == default_proto:
                        missing_seg_assigns += 1
                    y_proto[i] = int(proto)

        d_lab[p] = y_proto
        d_rid[p] = r

    print(f"[{method}] built prototype labels from {assignment_split_path.name}")
    if missing_map_proteins:
        print(f"  [warn] proteins missing from assignment_split: {missing_map_proteins}/{len(common_proteins)}")
    if missing_seg_assigns:
        print(f"  [warn] residues with segment_k not found in mapping: {missing_seg_assigns} (set to {default_proto})")

    return d_lab, d_rid


# ----------------------------
# WHERE IS assignment_split.csv?
# Put the correct path per method here.
# Example: "../ismb26/prototypes/puffin_K64_v4/K512/assignment_split.csv"
# ----------------------------
def assignment_split_path_for_method(method: str) -> Path:
    pe = Path(METHODS[method]["proto_enrich"])
    # proto_enrich: .../K512/eval/go_enrichment_train_all.csv
    # assignment_split.csv is usually in same folder:
    p = pe.parent/ "assignments_train.csv"
    if p.exists():
        return p
    # fallback: sometimes one level up
    p2 = pe.parent.parent / "assignments_train.csv"
    if p2.exists():
        return p2
    raise FileNotFoundError(f"Could not find assignments_train.csv for {method} near {pe}")


# --- Update loading: store both labels + residue ids per protein ---
method_dense = {}   # method -> dict(protein_key -> labels_on_sorted_resids)
method_resids = {}  # method -> dict(protein_key -> sorted_residue_ids)

for m, cfg in METHODS.items():
    adf = load_assignments_csv(cfg["assignments"])
    d_lab, d_rid = {}, {}
    for _, row in adf.iterrows():
        pk = str(row["protein_key"])
        lab, rids = assignment_to_labels_on_residue_ids(row)
        d_lab[pk] = lab
        d_rid[pk] = rids
    method_dense[m] = d_lab
    method_resids[m] = d_rid


# --- proteins common to all methods ---
common_proteins = sorted(set.intersection(*[set(d.keys()) for d in method_dense.values()]))
if len(common_proteins) == 0:
    raise ValueError("No overlapping proteins across methods. Check pdb_id -> legacy mapping.")
print(f"Common proteins across methods: {len(common_proteins)}")


# --- Helper: align labels to the intersection of residue-ids across methods (per protein) ---
def aligned_labels_for_protein(protein: str, methods: list[str]):
    # intersection of residue IDs
    sets = []
    for m in methods:
        r = method_resids[m][protein]
        sets.append(set(map(int, r.tolist())))

    inter = set.intersection(*sets)
    if len(inter) == 0:
        return None  # no overlap

    rid_common = np.array(sorted(inter), dtype=int)

    out = {}
    for m in methods:
        r = method_resids[m][protein]
        y = method_dense[m][protein]
        # map residue id -> label
        mp = {int(rr): int(yy) for rr, yy in zip(r.tolist(), y.tolist())}
        out[m] = np.array([mp[int(rr)] for rr in rid_common], dtype=int)

    return rid_common, out


def boundary_positions(labels: np.ndarray) -> np.ndarray:
    # boundaries at i where label changes between i-1 and i (i in [1..n-1])
    # ignore -1 gaps by treating them as labels too (still counts as boundary)
    if labels.size <= 1:
        return np.array([], dtype=int)
    return np.where(labels[1:] != labels[:-1])[0] + 1

def boundary_jaccard(b1: np.ndarray, b2: np.ndarray) -> float:
    s1, s2 = set(map(int, b1.tolist())), set(map(int, b2.tolist()))
    if not s1 and not s2:
        return 1.0
    if not s1 or not s2:
        return 0.0
    return len(s1 & s2) / len(s1 | s2)

def shannon_entropy(probs: np.ndarray, eps=1e-12) -> float:
    probs = probs[probs > 0]
    if probs.size == 0:
        return float("nan")
    return float(-np.sum(probs * np.log(probs + eps)))

def normalized_entropy_from_counts(counts: Counter) -> float:
    tot = sum(counts.values())
    if tot <= 0 or len(counts) <= 1:
        return 0.0 if tot > 0 else float("nan")
    p = np.array([v / tot for v in counts.values()], dtype=float)
    H = shannon_entropy(p)
    return float(H / np.log(len(counts)))


def mean_nearest_boundary_distance(b_src: np.ndarray, b_tgt: np.ndarray) -> float:
    # for each boundary in src, distance to nearest boundary in tgt
    if b_src.size == 0 or b_tgt.size == 0:
        return float("nan")
    b_src = np.asarray(b_src, dtype=int)
    b_tgt = np.sort(np.asarray(b_tgt, dtype=int))
    # vectorized nearest via searchsorted
    idx = np.searchsorted(b_tgt, b_src)
    left = np.clip(idx - 1, 0, b_tgt.size - 1)
    right = np.clip(idx, 0, b_tgt.size - 1)
    d = np.minimum(np.abs(b_src - b_tgt[left]), np.abs(b_src - b_tgt[right]))
    return float(np.mean(d))

# -----------------------
# 1) Prototype functional purity (GO entropy per prototype)
# -----------------------
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
from pathlib import Path


# Build prototype-space dicts for all methods that have proto_enrich
method_proto_dense = {}
method_proto_resids = {}

for m in METHODS:
    pe = METHODS[m].get("proto_enrich", None)
    if not pe or not Path(pe).exists():
        print(f"[skip] {m}: proto_enrich missing")
        continue
    asp = assignment_split_path_for_method(m)
    d_lab, d_rid = build_method_proto_dense_from_assignment_split(m, asp)
    method_proto_dense[m] = d_lab
    method_proto_resids[m] = d_rid

print("Prototype-space methods available:", list(method_proto_dense.keys()))


def prototype_go_entropy(method: str) -> pd.DataFrame:
    if method not in method_proto_dense:
        raise ValueError(f"{method}: prototype labels not available (build method_proto_dense first).")

    proto_to_prots = defaultdict(set)

    for p in common_proteins:
        y = method_proto_dense[method][p] 
        if y.size == 0:
            continue
        for proto in np.unique(y[y >= 0]):
            proto_to_prots[int(proto)].add(p)

    rows = []
    for proto, prots in proto_to_prots.items():
        term_counts = Counter()
        for p in prots:
            for t in prot_go.get(p, set()):
                term_counts[t] += 1
        Hn = normalized_entropy_from_counts(term_counts)
        rows.append({
            "method": method,
            "proto": int(proto),
            "n_proteins": int(len(prots)),
            "go_entropy_norm": float(Hn),
        })

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    # attach top enriched term label if proto_enrich exists
    pe = METHODS[method].get("proto_enrich", None)
    if pe and Path(pe).exists():
        enr = pd.read_csv(pe)
        key = "qval" if "qval" in enr.columns else "pval"
        enr[key] = pd.to_numeric(enr[key], errors="coerce")
        enr = enr.sort_values(["proto", key], ascending=[True, True])
        top = enr.groupby("proto", as_index=False).head(1)[["proto", "go_term", key]].copy()
        top = top.rename(columns={"go_term": "top_go_term", key: "top_q_or_p"})
        df = df.merge(top, on="proto", how="left")

    return df

# --- Update boundary shift computation to use aligned arrays ---
def boundary_shift_vs_baseline(method: str, baseline: str) -> pd.DataFrame:
    rows = []
    methods_here = [baseline, method]

    for p in common_proteins:
        aligned = aligned_labels_for_protein(p, methods_here)
        if aligned is None:
            continue
        rid_common, lab = aligned
        y0 = lab[baseline]
        y1 = lab[method]
        if y0.size == 0 or y1.size == 0:
            continue

        # residue-level changes in label on the common residue-id set
        frac_changed = float(np.mean(y0 != y1))

        # boundaries defined on the *ordered residue-id list* (by increasing residue id)
        b0 = boundary_positions(y0)
        b1 = boundary_positions(y1)
        jac = boundary_jaccard(b0, b1)
        mean_d01 = mean_nearest_boundary_distance(b1, b0)
        mean_d10 = mean_nearest_boundary_distance(b0, b1)

        rows.append({
            "protein": p,
            "method": method,
            "baseline": baseline,
            "n_positions_common": int(rid_common.size),
            "frac_residues_changed": frac_changed,
            "n_boundaries_baseline": int(b0.size),
            "n_boundaries_method": int(b1.size),
            "boundary_jaccard": float(jac),
            "mean_boundary_dist_method_to_base": float(mean_d01),
            "mean_boundary_dist_base_to_method": float(mean_d10),
        })
    return pd.DataFrame(rows)

go_df = load_go_tsv(GO_TSV)
prot_go = {pid: set(safe_go_list(go_df.loc[pid, GO_ASPECT])) for pid in go_df.index.astype(str)}


entropy_dfs = []
for m in METHODS:
    entropy_dfs.append(prototype_go_entropy(m))
entropy_df = pd.concat(entropy_dfs, ignore_index=True)

# --- Recompute shift_df with the new function ---
shift_dfs = []
for m in METHODS:
    if m == STRUCT_BASELINE:
        continue
    shift_dfs.append(boundary_shift_vs_baseline(m, STRUCT_BASELINE))
shift_df = pd.concat(shift_dfs, ignore_index=True)

print("Shift df:", shift_df.shape)
display(shift_df.head())


In [ ]:

fig = plt.figure(figsize=(12.5, 9.6))
gs = fig.add_gridspec(2, 2, height_ratios=[1, 1.05], width_ratios=[1, 1], hspace=0.32, wspace=0.22)

# A: entropy
axA = fig.add_subplot(gs[0, 0])
dataA, labelsA = [], []
for m, cfg in METHODS.items():
    v = entropy_df.loc[entropy_df["method"] == m, "go_entropy_norm"].to_numpy(dtype=float)
    v = v[np.isfinite(v)]
    if v.size:
        dataA.append(v); labelsA.append(cfg["label"])
axA.violinplot(dataA, showmeans=False, showmedians=True, showextrema=False)
axA.boxplot(dataA, widths=0.15, showfliers=False)
axA.set_xticks(np.arange(1, len(labelsA) + 1))
axA.set_xticklabels(labelsA, rotation=20, ha="right")
axA.set_ylabel("Normalized GO entropy per prototype (↓ purer)")
axA.set_title("A  Prototype functional purity")

# B: reuse (#proteins per prototype)
axB = fig.add_subplot(gs[0, 1])
dataB, labelsB = [], []
for m, cfg in METHODS.items():
    v = entropy_df.loc[entropy_df["method"] == m, "n_proteins"].to_numpy(dtype=float)
    v = v[np.isfinite(v)]
    if v.size:
        dataB.append(v); labelsB.append(cfg["label"])
axB.violinplot(dataB, showmeans=False, showmedians=True, showextrema=False)
axB.boxplot(dataB, widths=0.15, showfliers=False)
axB.set_xticks(np.arange(1, len(labelsB) + 1))
axB.set_xticklabels(labelsB, rotation=20, ha="right")
axB.set_ylabel("# proteins contributing to prototype (reuse)")
axB.set_title("B  Prototype reuse across proteins")

# C: boundary shift (two subplots)
gsC = gs[1, :].subgridspec(1, 2, wspace=0.25)

# axC1 = fig.add_subplot(gsC[0, 0])
# axC2 = fig.add_subplot(gsC[0, 1])

# C1: fraction residues changed
# dataC1, labelsC = [], []
# for m, cfg in METHODS.items():
#     if m == STRUCT_BASELINE:
#         continue
#     v = shift_df.loc[shift_df["method"] == m, "frac_residues_changed"].to_numpy(dtype=float)
#     v = v[np.isfinite(v)]
#     if v.size:
#         dataC1.append(v); labelsC.append(cfg["label"])
# axC1.violinplot(dataC1, showmeans=False, showmedians=True, showextrema=False)
# axC1.boxplot(dataC1, widths=0.15, showfliers=False)
# axC1.set_xticks(np.arange(1, len(labelsC) + 1))
# axC1.set_xticklabels(labelsC, rotation=20, ha="right")
# axC1.set_ylabel("Fraction residues with different assignment vs MinCut")
# axC1.set_title("C1  Assignment shift vs struct-only baseline")

# # C2: boundary jaccard
# dataC2 = []
# for m, cfg in METHODS.items():
#     if m == STRUCT_BASELINE:
#         continue
#     v = shift_df.loc[shift_df["method"] == m, "boundary_jaccard"].to_numpy(dtype=float)
#     v = v[np.isfinite(v)]
#     if v.size:
#         dataC2.append(v)
# axC2.violinplot(dataC2, showmeans=False, showmedians=True, showextrema=False)
# axC2.boxplot(dataC2, widths=0.15, showfliers=False)
# axC2.set_xticks(np.arange(1, len(labelsC) + 1))
# axC2.set_xticklabels(labelsC, rotation=20, ha="right")
# axC2.set_ylabel("Boundary Jaccard vs MinCut (↑ more similar)")
# axC2.set_title("C2  Boundary preservation vs struct-only baseline")

# plt.show()

# Optional: quick summary table (method-level)
summary = []
for m, cfg in METHODS.items():
    ent = entropy_df.loc[entropy_df["method"] == m, "go_entropy_norm"]
    reuse = entropy_df.loc[entropy_df["method"] == m, "n_proteins"]
    row = {
        "method": cfg["label"],
        "proto_go_entropy_median": float(np.nanmedian(ent)) if len(ent) else np.nan,
        "proto_go_entropy_mean": float(np.nanmean(ent)) if len(ent) else np.nan,
        "proto_reuse_median": float(np.nanmedian(reuse)) if len(reuse) else np.nan,
        "proto_reuse_mean": float(np.nanmean(reuse)) if len(reuse) else np.nan,
        "proto_reuse_min": float(np.nanmin(reuse)) if len(reuse) else np.nan,
        "proto_reuse_max": float(np.nanmax(reuse)) if len(reuse) else np.nan
    }
    if m != STRUCT_BASELINE:
        sh = shift_df[shift_df["method"] == m]
        row.update({
            "frac_residues_changed_median": float(np.nanmedian(sh["frac_residues_changed"])) if len(sh) else np.nan,
            "boundary_jaccard_median": float(np.nanmedian(sh["boundary_jaccard"])) if len(sh) else np.nan,
            "mean_boundary_dist_method_to_base": float(np.nanmean(sh["mean_boundary_dist_method_to_base"])) if len(sh) else np.nan,
        })
    summary.append(row)

display(pd.DataFrame(summary))

In [ ]:
# Notebook cell: Protein-conditioned prototype GO enrichment + lift + leakage + negative controls
#
# Goal:
#   - For each prototype k, compute protein-conditioned empirical p-values (permutation) for GO terms
#   - Report lift beyond protein composition ("annotation co-occurrence" control)
#   - Report an annotation leakage score (how much of signal is explained by protein set alone)
#   - Provide "hard" negative controls:
#       (NC1) shuffle proto labels within each protein (preserve per-protein proto histogram)
#       (NC2) shuffle proto labels globally (preserve proto sizes)
#       (NC3) protein-conditioned permutation itself is the main control (matched proteins)
#
# Requirements (already in your notebook/script):
#   - go_df indexed by legacy id with columns MF/BP/CC (loaded via load_go_tsv/load_go_annotations)
#   - safe_go_list(...)
#   - GO_ASPECT = "MF" (or BP/CC)
#
# Input data:
#   - segment-level assignments CSV (recommended), e.g. assignments_train.csv from your global prototype pipeline:
#       columns include: global_seg_index, pdb_id, legacy_id, protein_key, segment_k, n_residues_assigned, proto, assign_sim
#
# NOTE: This cell does NOT use residue-level "runs". It uses true global prototypes from assignments_* CSV.
#
# This replaces the "Load assignments (segment-level; prototypes ...)" block from my previous cell.
# It uses your pipeline outputs next to proto_enrich:
#   .../K{K}/eval/go_enrichment_train_all.csv
#   .../K{K}/assignments_train.csv          (or one level up)
#
# It assumes you already defined:
#   - METHODS, METHOD, K
#   - assignment_split_path_for_method(method)
#   - load_assignment_split(path)
#   - go_df, GO_ASPECT, safe_go_list

from pathlib import Path
import numpy as np
import pandas as pd

# Which split file to use (train/valid/test)
SPLIT = "train"  # change to "valid" or "test" if you want
METHOD = "puffin_v4"  # change to your method of choice

def assignment_split_path_for_method_and_split(method: str, split: str) -> Path:
    """
    Finds assignments_{split}.csv near proto_enrich.
    Your current helper returns assignments_train.csv; this one generalizes.
    """
    pe = Path(METHODS[method]["proto_enrich"])
    # pe: .../K{K}/eval/go_enrichment_{split}_all.csv
    # assignments: .../K{K}/assignments_{split}.csv (most common)
    p = pe.parent.parent / f"assignments_{split}.csv"   # .../K{K}/assignments_train.csv
    if p.exists():
        return p
    # fallback: sometimes in eval/
    p2 = pe.parent / f"assignments_{split}.csv"         # .../K{K}/eval/assignments_train.csv
    if p2.exists():
        return p2
    # fallback: sometimes one level up (rare)
    p3 = pe.parent.parent.parent / f"assignments_{split}.csv"
    if p3.exists():
        return p3
    raise FileNotFoundError(f"Could not find assignments_{split}.csv for {method} near {pe}")

# 1) load assignment_split / assignments_train.csv (segment-level assignments to global prototypes)
assign_path = assignment_split_path_for_method_and_split(METHOD, SPLIT)
assign_df = load_assignment_split(assign_path)

# Expected columns from your global_prototypes_splitfit outputs:
# protein_key, segment_k, proto  (+ optional: global_seg_index, pdb_id, legacy_id, n_residues_assigned, assign_sim)
# If "protein_key" is missing (unlikely), we can reconstruct from legacy_id.
if "protein_key" not in assign_df.columns:
    if "legacy_id" in assign_df.columns:
        assign_df["protein_key"] = assign_df["legacy_id"].astype(str)
    else:
        raise ValueError(f"{assign_path} has no protein_key (and no legacy_id to reconstruct it)")

assign_df["protein_key"] = assign_df["protein_key"].astype(str)
assign_df["proto"] = pd.to_numeric(assign_df["proto"], errors="coerce").astype(int)

# 2) (optional) keep only proteins that exist in GO table for the chosen aspect
go_col = go_df[GO_ASPECT]
go_index = set(go_col.index.astype(str).tolist())
assign_df = assign_df[assign_df["protein_key"].isin(go_index)].copy()

# 3) build protein->GO sets
prot_go = {pid: set(safe_go_list(go_col.loc[pid])) for pid in go_col.index.astype(str)}

# 4) build proto->proteins mapping (THIS is what enrichment/permutation uses)
proto_to_prots = (
    assign_df.groupby("proto")["protein_key"]
    .apply(lambda x: set(x.astype(str).tolist()))
    .to_dict()
)

print(f"[{METHOD}] loaded {assign_path.name} ({SPLIT})")
print("assign_df:", assign_df.shape,
      "| proteins:", assign_df['protein_key'].nunique(),
      "| protos:", assign_df['proto'].nunique())

# ---- If you want to reuse the length-matching null from my previous cell ----
# It needs prot_to_seg_idx and n_residues_assigned. We'll set safe defaults.
if "n_residues_assigned" in assign_df.columns:
    assign_df["n_residues_assigned"] = pd.to_numeric(assign_df["n_residues_assigned"], errors="coerce").fillna(0).astype(int)
else:
    # if not available, length-matching will degrade to "any segment in protein"
    assign_df["n_residues_assigned"] = 0

prot_to_seg_idx = {p: g.index.to_numpy() for p, g in assign_df.groupby("protein_key", sort=False)}


# Optional: proto->proteins set (true global prototypes)
proto_to_prots = assign_df.groupby("proto")["protein_key"].apply(lambda x: set(x.astype(str).tolist())).to_dict()

# ----------------------------
# Helpers
# ----------------------------
def _term_counts_for_proteins(prots):
    c = Counter()
    for p in prots:
        for t in prot_go.get(p, set()):
            c[t] += 1
    return c

def _obs_counts_for_proto(proto_id: int):
    prots = proto_to_prots.get(int(proto_id), set())
    n = len(prots)
    if n == 0:
        return set(), Counter()
    return prots, _term_counts_for_proteins(prots)

def _sample_length_matched_segment_from_protein(p: str, target_len: int, rng: np.random.Generator, tol: int = 2):
    """
    Choose one segment from protein p with n_residues_assigned close to target_len.
    If none exist, fall back to any segment from that protein.
    Returns: chosen protein_key (same p) (segment identity doesn't matter for GO, only protein is used),
             but we keep this function to support length-matching and future segment-local proxies.
    """
    idxs = prot_to_seg_idx.get(p, None)
    if idxs is None or len(idxs) == 0:
        return None
    sub = assign_df.loc[idxs, ["n_residues_assigned"]]
    L = sub["n_residues_assigned"].to_numpy(int)

    # match within tol
    ok = np.where(np.abs(L - int(target_len)) <= int(tol))[0]
    if ok.size > 0:
        pick_local = int(rng.choice(ok))
        pick = int(idxs[pick_local])
        return pick
    # fallback any
    pick = int(rng.choice(idxs))
    return pick

def protein_conditioned_permutation_for_proto(
    proto_id: int,
    *,
    n_perm: int = 2000,
    seed: int = 0,
    length_match_tol: int = 2,
    use_protein_restricted_null: bool = True,
):
    """
    Protein-conditioned permutation:
      - Keep the prototype's *protein set* fixed (S_k).
      - For each protein, draw a random segment from that protein with length matched to the
        prototype member segment length distribution (coarse; per-protein target = median length of that protein's proto-members).
      - Null statistic for GO term t: #proteins in sampled set with term t.
    Returns:
      obs_counts: Counter term->a_obs
      null_counts: dict term->np.array shape (n_perm,) with a_null
      meta: dict
    """
    rng = np.random.default_rng(int(seed))
    proto_id = int(proto_id)

    # protein set for this prototype
    S = sorted(list(proto_to_prots.get(proto_id, set())))
    if len(S) == 0:
        return Counter(), {}, {"proto": proto_id, "n_proteins": 0}

    # For each protein, set a target length based on proto membership segments in that protein
    # (if you have multiple segments in the same proto+protein, use median).
    dfk = assign_df[assign_df["proto"] == proto_id]
    # group lengths by protein (only among segments assigned to this proto)
    prot_target_len = (
        dfk.groupby("protein_key")["n_residues_assigned"]
        .median()
        .reindex(S)
        .fillna(dfk["n_residues_assigned"].median() if len(dfk) > 0 else 10)
        .astype(int)
        .to_dict()
    )

    # Observed term counts among proteins S
    obs_counts = _term_counts_for_proteins(S)

    # Terms to track in null:
    # - Only track terms that appear at least once in observed set (keeps arrays small)
    terms = sorted(list(obs_counts.keys()))
    null_mat = {t: np.zeros((int(n_perm),), dtype=int) for t in terms}

    # Permute
    for r in range(int(n_perm)):
        sampled_prots = []

        if use_protein_restricted_null:
            # sample one segment per protein FROM THE SAME protein (preserves protein set)
            for p in S:
                tL = int(prot_target_len.get(p, 10))
                seg_row_idx = _sample_length_matched_segment_from_protein(p, tL, rng, tol=length_match_tol)
                if seg_row_idx is None:
                    continue
                # protein identity stays p; we append p (not seg id) since GO is protein-level here
                sampled_prots.append(p)
        else:
            # alternative null: sample proteins from the whole dataset matched in size
            # (we keep this option but default is True)
            all_prots = np.array(sorted(list(prot_to_seg_idx.keys())), dtype=object)
            sampled_prots = rng.choice(all_prots, size=len(S), replace=False).tolist()

        # compute null counts for tracked terms
        # (fast: update by iterating proteins once)
        c = Counter()
        for p in sampled_prots:
            for t in prot_go.get(p, set()):
                if t in null_mat:
                    c[t] += 1
        for t in terms:
            null_mat[t][r] = int(c.get(t, 0))

    meta = {
        "proto": proto_id,
        "n_proteins": int(len(S)),
        "n_perm": int(n_perm),
        "seed": int(seed),
        "length_match_tol": int(length_match_tol),
    }
    return obs_counts, null_mat, meta

def summarize_protein_conditioned_enrichment(proto_id: int, *, n_perm: int = 2000, seed: int = 0, top_n: int = 20):
    """
    Returns DataFrame with:
      - a_obs, frac_obs
      - null_mean, null_frac_mean
      - p_emp (one-sided: >=)
      - lift_log2 (obs_frac / null_frac_mean)
      - leakage_score (null_mean / obs)  in [0,1+] ; near 1 means "explained by co-occurrence"
    """
    obs_counts, null_mat, meta = protein_conditioned_permutation_for_proto(proto_id, n_perm=n_perm, seed=seed)

    nP = meta["n_proteins"]
    if nP == 0:
        return pd.DataFrame()

    rows = []
    for t, a_obs in obs_counts.items():
        null = null_mat.get(t, None)
        if null is None:
            continue
        null_mean = float(np.mean(null))
        # empirical p-value (>= observed)
        p_emp = float((1.0 + np.sum(null >= int(a_obs))) / (1.0 + null.size))

        obs_frac = float(a_obs / max(1, nP))
        null_frac = float(null_mean / max(1, nP))

        lift_log2 = float(np.log2((obs_frac + EPS) / (null_frac + EPS)))

        # leakage_score: how much of observed is explained by protein composition alone
        #  ~1 => almost fully explained (likely co-occurrence)
        #  ~0 => null expectation near 0 but observed positive (less leakage)
        leakage = float(null_mean / max(EPS, float(a_obs)))

        rows.append({
            "proto": int(proto_id),
            "go_term": t,
            "a_obs": int(a_obs),
            "frac_obs": obs_frac,
            "null_mean": null_mean,
            "null_frac_mean": null_frac,
            "p_emp": p_emp,
            "lift_log2": lift_log2,
            "leakage_score": leakage,
        })

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    # sort: strongest lift, then smallest p
    df = df.sort_values(["lift_log2", "p_emp", "a_obs"], ascending=[False, True, False]).reset_index(drop=True)

    # also give a prototype-level leakage summary (top terms)
    top = df.head(int(top_n)).copy()
    proto_leak = float(np.median(np.clip(top["leakage_score"].to_numpy(float), 0, 10))) if len(top) else float("nan")
    print(f"[proto {proto_id}] proteins={nP} | median leakage (top{top_n}) = {proto_leak:.3f}")
    return df

# ----------------------------
# "Hard" negative controls
# ----------------------------
def nc_shuffle_within_protein(assign_df_in: pd.DataFrame, seed: int = 0) -> pd.DataFrame:
    """
    Shuffle proto labels within each protein.
    Preserves per-protein proto histogram and proto sizes within that protein.
    Destroys any true segment-to-proto structure, keeps co-occurrence patterns.
    """
    rng = np.random.default_rng(int(seed))
    df = assign_df_in.copy()
    new_proto = df["proto"].to_numpy(int).copy()

    for p, idxs in prot_to_seg_idx.items():
        idxs = np.asarray(idxs, dtype=int)
        if idxs.size <= 1:
            continue
        # shuffle proto labels only among this protein's segments
        prot_protos = new_proto[idxs].copy()
        rng.shuffle(prot_protos)
        new_proto[idxs] = prot_protos

    df["proto"] = new_proto
    return df

def nc_shuffle_global(assign_df_in: pd.DataFrame, seed: int = 0) -> pd.DataFrame:
    """
    Shuffle proto labels globally (preserves proto size distribution, destroys protein structure).
    """
    rng = np.random.default_rng(int(seed))
    df = assign_df_in.copy()
    protos = df["proto"].to_numpy(int).copy()
    rng.shuffle(protos)
    df["proto"] = protos
    return df

def evaluate_proto_term_under_assignment_df(proto_id: int, assign_df_custom: pd.DataFrame, *, n_perm: int = 500, seed: int = 0):
    """
    Run the same protein-conditioned test but using a custom assign_df (e.g. shuffled labels).
    """
    global assign_df, prot_to_seg_idx, proto_to_prots
    # swap globals temporarily (simple + notebook-friendly)
    _assign_df = assign_df
    _prot_to_seg_idx = prot_to_seg_idx
    _proto_to_prots = proto_to_prots
    try:
        assign_df = assign_df_custom
        prot_to_seg_idx = {p: g.index.to_numpy() for p, g in assign_df.groupby("protein_key", sort=False)}
        proto_to_prots = assign_df.groupby("proto")["protein_key"].apply(lambda x: set(x.astype(str).tolist())).to_dict()
        return summarize_protein_conditioned_enrichment(proto_id, n_perm=n_perm, seed=seed, top_n=20)
    finally:
        assign_df = _assign_df
        prot_to_seg_idx = _prot_to_seg_idx
        proto_to_prots = _proto_to_prots

# ----------------------------
# Run for a prototype
# ----------------------------
import random
N_PERM = 2000
SEED = 0
PROTO_ID = random.choice(assign_df["proto"].unique().tolist())
df_pc = summarize_protein_conditioned_enrichment(PROTO_ID, n_perm=N_PERM, seed=SEED, top_n=20)
display(df_pc.head(25))

# Quick visualization: top terms by lift
if not df_pc.empty:
    top = df_pc.head(15).iloc[::-1]
    plt.figure(figsize=(7.5, 4.2))
    plt.barh(top["go_term"], top["lift_log2"])
    plt.xlabel("Protein-conditioned lift (log2)")
    plt.title(f"{METHOD} proto {PROTO_ID}: GO lift beyond protein composition ({GO_ASPECT})")
    plt.tight_layout()
    plt.show()

# ----------------------------
# Negative controls (sanity checks)
# ----------------------------
NC_PERM = 800  # keep smaller for speed

print("\n[NC1] Shuffle proto labels within protein")
df_nc1 = evaluate_proto_term_under_assignment_df(PROTO_ID, nc_shuffle_within_protein(assign_df, seed=SEED+1), n_perm=NC_PERM, seed=SEED+11)
display(df_nc1.head(10))

print("\n[NC2] Shuffle proto labels globally")
df_nc2 = evaluate_proto_term_under_assignment_df(PROTO_ID, nc_shuffle_global(assign_df, seed=SEED+2), n_perm=NC_PERM, seed=SEED+22)
display(df_nc2.head(10))

def summarize_nc(df, name):
    if df is None or df.empty:
        print(f"{name}: empty")
        return
    # if there is real prototype signal, NC should push lifts ~0 and p_emp ~uniform
    print(f"{name}: max lift={df['lift_log2'].max():.3f}, median lift={df['lift_log2'].median():.3f}, min p={df['p_emp'].min():.4f}")

summarize_nc(df_pc, "OBS")
summarize_nc(df_nc1, "NC1 within-protein shuffle")
summarize_nc(df_nc2, "NC2 global shuffle")


In [ ]:
# Notebook cell: compute prototype-level stats for *ALL* proto ids, save, and summarize
#
# Uses the corrected prototype loader (assignments_{split}.csv next to proto_enrich)
# and your GO TSV (protein_key is legacy id).
#
# Output (per method):
#   - <OUT_DIR>/<method>__proto_characterization_<split>.csv
#   - <OUT_DIR>/<method>__proto_summary_<split>.csv
#
# What we compute per prototype:
#   - n_segments (rows in assignments_{split}.csv)
#   - n_proteins (unique proteins contributing segments to this prototype)
#   - GO functional profile entropy (normalized) across contributing proteins
#   - top GO term label (from proto_enrich) if available
#   - assignment similarity stats (mean/median) if assign_sim exists
#   - segment length stats (mean/median) if n_residues_assigned exists

from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd

# -------------------
# USER CONTROLS
# -------------------
SPLIT = "train"  # {"train","valid","test"} depending on what you want
OUT_DIR = Path("./proto_characterization_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# -------------------
# helpers
# -------------------
def normalized_entropy_from_counts(counts: Counter) -> float:
    tot = sum(counts.values())
    if tot <= 0:
        return float("nan")
    if len(counts) <= 1:
        return 0.0
    p = np.array([v / tot for v in counts.values()], dtype=float)
    H = float(-np.sum(p * np.log(p + 1e-12)))
    return float(H / np.log(len(counts)))

def assignment_split_path_for_method_and_split(method: str, split: str) -> Path:
    pe = Path(METHODS[method]["proto_enrich"])
    # pe: .../K{K}/eval/go_enrichment_{split}_all.csv
    p = pe.parent.parent / f"assignments_{split}.csv"   # .../K{K}/assignments_train.csv
    if p.exists():
        return p
    p2 = pe.parent / f"assignments_{split}.csv"         # .../K{K}/eval/assignments_train.csv
    if p2.exists():
        return p2
    p3 = pe.parent.parent.parent / f"assignments_{split}.csv"
    if p3.exists():
        return p3
    raise FileNotFoundError(f"Could not find assignments_{split}.csv for {method} near {pe}")

def load_proto_top_terms(method: str, split: str) -> pd.DataFrame:
    """
    Reads proto_enrich (go_enrichment_{split}_all.csv) and returns 1 row per proto
    with best (lowest qval or pval) term.
    """
    pe = Path(METHODS[method].get("proto_enrich", ""))
    if not pe.exists():
        return pd.DataFrame(columns=["proto", "top_go_term", "top_q_or_p"])

    # If your proto_enrich is always TRAIN, we keep it (still useful labels).
    # If you also have valid/test enrich files, you can swap path logic here.
    enr = pd.read_csv(pe)
    if "proto" not in enr.columns or "go_term" not in enr.columns:
        return pd.DataFrame(columns=["proto", "top_go_term", "top_q_or_p"])

    key = "qval" if "qval" in enr.columns else ("pval" if "pval" in enr.columns else None)
    if key is None:
        return pd.DataFrame(columns=["proto", "top_go_term", "top_q_or_p"])

    enr[key] = pd.to_numeric(enr[key], errors="coerce")
    enr = enr.dropna(subset=["proto", "go_term", key]).copy()
    if enr.empty:
        return pd.DataFrame(columns=["proto", "top_go_term", "top_q_or_p"])

    enr["proto"] = pd.to_numeric(enr["proto"], errors="coerce").astype(int)
    enr = enr.sort_values(["proto", key], ascending=[True, True])
    top = enr.groupby("proto", as_index=False).head(1)[["proto", "go_term", key]].copy()
    top = top.rename(columns={"go_term": "top_go_term", key: "top_q_or_p"})
    return top

import json

def characterize_all_protos_for_method(method: str, split: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    assign_path = assignment_split_path_for_method_and_split(method, split)
    assign_df = load_assignment_split(assign_path)

    if "protein_key" not in assign_df.columns:
        if "legacy_id" in assign_df.columns:
            assign_df["protein_key"] = assign_df["legacy_id"].astype(str)
        else:
            raise ValueError(f"{assign_path} missing protein_key and legacy_id")

    assign_df["protein_key"] = assign_df["protein_key"].astype(str)
    assign_df["proto"] = pd.to_numeric(assign_df["proto"], errors="coerce")
    assign_df = assign_df.dropna(subset=["proto"]).copy()
    assign_df["proto"] = assign_df["proto"].astype(int)

    # keep proteins that exist in GO table
    go_col = go_df[GO_ASPECT]
    go_index = set(go_col.index.astype(str).tolist())
    assign_df = assign_df[assign_df["protein_key"].isin(go_index)].copy()

    # optional numeric columns
    if "assign_sim" in assign_df.columns:
        assign_df["assign_sim"] = pd.to_numeric(assign_df["assign_sim"], errors="coerce")
    if "n_residues_assigned" in assign_df.columns:
        assign_df["n_residues_assigned"] = pd.to_numeric(assign_df["n_residues_assigned"], errors="coerce")

    proto_to_prots = (
        assign_df.groupby("proto")["protein_key"]
        .apply(lambda x: set(x.astype(str).tolist()))
        .to_dict()
    )

    rows = []
    for proto, prots in proto_to_prots.items():
        term_counts = Counter()
        for p in prots:
            for t in safe_go_list(go_col.loc[p]):
                term_counts[t] += 1
        go_entropy_norm = normalized_entropy_from_counts(term_counts)

        sub = assign_df[assign_df["proto"] == int(proto)]
        n_segments = int(len(sub))
        n_proteins = int(len(prots))

        row = {
            "method": method,
            "method_label": METHODS[method].get("label", method),
            "split": split,
            "proto": int(proto),
            "n_segments": n_segments,
            "n_proteins": n_proteins,
            "go_entropy_norm": float(go_entropy_norm),
            "n_go_terms_observed": int(len(term_counts)),
        }

        if "assign_sim" in sub.columns:
            x = sub["assign_sim"].dropna().astype(float).to_numpy()
            row["assign_sim_mean"] = float(np.mean(x)) if x.size else float("nan")
            row["assign_sim_median"] = float(np.median(x)) if x.size else float("nan")

        if "n_residues_assigned" in sub.columns:
            L = sub["n_residues_assigned"].dropna().astype(float).to_numpy()
            row["seglen_mean"] = float(np.mean(L)) if L.size else float("nan")
            row["seglen_median"] = float(np.median(L)) if L.size else float("nan")
            row["seglen_p10"] = float(np.quantile(L, 0.10)) if L.size else float("nan")
            row["seglen_p90"] = float(np.quantile(L, 0.90)) if L.size else float("nan")

        rows.append(row)

    per_proto_df = pd.DataFrame(rows)
    if per_proto_df.empty:
        return per_proto_df, pd.DataFrame()

    # attach top enriched GO term label if proto_enrich exists
    top_terms = load_proto_top_terms(method, split=split)
    if not top_terms.empty:
        per_proto_df = per_proto_df.merge(top_terms, on="proto", how="left")

    # ---- FIX: build a guaranteed 1-row summary df (no pandas agg tuple weirdness) ----
    top_support = per_proto_df.sort_values(["n_proteins", "n_segments"], ascending=[False, False]).head(5)
    top_pure = per_proto_df.sort_values(["go_entropy_norm", "n_proteins"], ascending=[True, False]).head(5)

    # store as JSON strings so CSV writing/reading is robust
    top_support_json = json.dumps(
        top_support[["proto", "n_proteins", "n_segments", "top_go_term", "top_q_or_p"]]
        .fillna("")
        .to_dict(orient="records")
    )
    top_pure_json = json.dumps(
        top_pure[["proto", "go_entropy_norm", "n_proteins", "top_go_term", "top_q_or_p"]]
        .fillna("")
        .to_dict(orient="records")
    )

    summary_df = pd.DataFrame([{
        "method": method,
        "method_label": METHODS[method].get("label", method),
        "split": split,
        "n_protos": int(per_proto_df["proto"].nunique()),
        "n_segments_total": int(per_proto_df["n_segments"].sum()),
        "go_entropy_mean": float(per_proto_df["go_entropy_norm"].mean()),
        "go_entropy_median": float(per_proto_df["go_entropy_norm"].median()),
        "top5_supporting_protos_json": top_support_json,
        "top5_pure_protos_json": top_pure_json,
    }])

    return per_proto_df, summary_df

# -------------------
# RUN: all methods
# -------------------
all_per_proto = []
all_summaries = []

for m in METHODS:
    # require that proto_enrich exists (otherwise we may not have assignments_{split}.csv nearby)
    pe = Path(METHODS[m].get("proto_enrich", ""))
    if not pe.exists():
        print(f"[skip] {m}: proto_enrich missing (can't locate assignments_{SPLIT}.csv reliably)")
        continue

    try:
        per_proto_df, summary_df = characterize_all_protos_for_method(m, split=SPLIT)
        if per_proto_df.empty:
            print(f"[warn] {m}: no proto rows after filtering")
            continue

        # save per-method
        out_csv = OUT_DIR / f"{m}__proto_characterization_{SPLIT}.csv"
        out_sum = OUT_DIR / f"{m}__proto_summary_{SPLIT}.csv"
        per_proto_df.to_csv(out_csv, index=False)
        summary_df.to_csv(out_sum, index=False)

        print(f"[ok] {m}: wrote {out_csv.name} ({len(per_proto_df)} protos)")
        all_per_proto.append(per_proto_df)
        all_summaries.append(summary_df)

    except Exception as e:
        print(f"[fail] {m}: {e}")

# combined outputs across methods
if all_per_proto:
    all_proto_df = pd.concat(all_per_proto, ignore_index=True)
    all_proto_df.to_csv(OUT_DIR / f"ALL__proto_characterization_{SPLIT}.csv", index=False)
    display(all_proto_df.head(10))

if all_summaries:
    all_sum_df = pd.concat(all_summaries, ignore_index=True)
    all_sum_df.to_csv(OUT_DIR / f"ALL__proto_summary_{SPLIT}.csv", index=False)
    display(all_sum_df)


In [ ]:
# Population-level semantic alignment (prototype enrichment, global MF signal)
# Reports (per method):
#   - mean enrichment odds ratio (among enriched prototypes)
#   - % enriched prototypes (FDR/qval < threshold)
#   - median GO depth (and median IC) of each prototype's top enriched MF term
#   - mean ± 95% CI (bootstrap over prototypes)

import numpy as np
import pandas as pd
from pathlib import Path

# ---------------------------
# USER PARAMS
# ---------------------------
FDR_THR = 0.05                # enriched prototype threshold
N_BOOT = 2000                 # bootstrap resamples for CI
SEED = 0


# Optional: for GO depth/IC
GO_OBO = Path("../data/go-basic-2020-06-01.obo")  # change if needed
GO_TSV = Path("../data/GeneOntology/nrPDB-GO_annot.tsv")  # used for IC from term frequencies

# ---------------------------
# GO annotation TSV loader (same skiprows trick)
# ---------------------------
def _load_go_tsv(path: Path) -> pd.DataFrame:
    for skip in (12, 14, 13):
        try:
            df = pd.read_csv(path, sep="\t", skiprows=skip)
            if df.shape[1] >= 4:
                df = df.iloc[:, :4].copy()
                df.columns = ["PDB", "MF", "BP", "CC"]
                df["PDB"] = df["PDB"].astype(str)
                return df.set_index("PDB")
        except Exception:
            pass
    raise ValueError(f"Could not parse GO TSV: {path}")

def _safe_go_list(val):
    if val is None:
        return []
    if isinstance(val, float) and np.isnan(val):
        return []
    s = str(val).strip()
    if not s or s.lower() == "nan":
        return []
    return [t for t in s.split(",") if t]

# ---------------------------
# Depth + IC helpers 
#   - Depth via GO DAG if goatools + go-basic.obo are available
#   - IC via -log(freq) from GO TSV frequencies in chosen aspect (no propagation)
# ---------------------------
def load_go_dag(obo_path: Path):
    from goatools.obo_parser import GODag
    if not obo_path.exists():
        return None
    return GODag(str(obo_path), optional_attrs={"depth"})
    

def _term_depth(term: str, godag) -> float:
    if godag is None:
        return np.nan
    t = godag.get(term, None)
    if t is None:
        return np.nan
    # goatools usually fills .depth, but be defensive
    d = getattr(t, "depth", None)
    return float(d) if d is not None else np.nan

def _build_ic_from_go_tsv(go_tsv_path: Path, aspect: str):
    if not go_tsv_path.exists():
        return None
    df = _load_go_tsv(go_tsv_path)
    col = df[aspect.upper()]
    counts = {}
    total = 0
    for pid in col.index.astype(str):
        terms = _safe_go_list(col.loc[pid])
        for t in terms:
            counts[t] = counts.get(t, 0) + 1
            total += 1
    if total == 0:
        return None
    # IC = -log(p); p = count/total term occurrences (no propagation)
    ic = {t: float(-np.log(c / total)) for t, c in counts.items() if c > 0}
    return ic

# ---------------------------
# Bootstrap CI helper
# ---------------------------
def _bootstrap_mean_ci(x: np.ndarray, n_boot: int = 2000, seed: int = 0):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    n = x.size
    boots = np.empty(n_boot, dtype=float)
    for b in range(n_boot):
        samp = rng.choice(x, size=n, replace=True)
        boots[b] = np.mean(samp)
    mu = float(np.mean(x))
    lo, hi = np.quantile(boots, [0.025, 0.975])
    return mu, float(lo), float(hi)

# ---------------------------
# Main: build Q3 table
# ---------------------------
godag = load_go_dag(GO_OBO)
ic_map = _build_ic_from_go_tsv(GO_TSV, GO_ASPECT)

rows = []
per_method_details = {}  # keep if you want to inspect later

for method, cfg in METHODS.items():
    pe_path = Path(cfg.get("proto_enrich", ""))
    if not pe_path.exists():
        print(f"[WARN] Missing proto_enrich for {method}: {pe_path}")
        continue

    enr = pd.read_csv(pe_path)
    # normalize columns
    if "go_term" not in enr.columns:
        raise ValueError(f"{pe_path} missing 'go_term'")
    if "proto" not in enr.columns:
        raise ValueError(f"{pe_path} missing 'proto'")
    if "qval" not in enr.columns and "pval" not in enr.columns:
        raise ValueError(f"{pe_path} needs 'qval' (preferred) or 'pval'")

    score_col = "qval" if "qval" in enr.columns else "pval"
    odds_col = None
    for cand in ["odds_ratio_approx", "odds_ratio", "odds", "OR"]:
        if cand in enr.columns:
            odds_col = cand
            break
    if odds_col is None:
        raise ValueError(f"{pe_path} missing odds ratio column (expected odds_ratio_approx)")

    # one top term per prototype = smallest q/p
    enr = enr.copy()
    enr[score_col] = pd.to_numeric(enr[score_col], errors="coerce")
    enr[odds_col] = pd.to_numeric(enr[odds_col], errors="coerce")

    top = (
        enr.sort_values(["proto", score_col], ascending=[True, True])
           .groupby("proto", as_index=False)
           .head(1)
           .reset_index(drop=True)
    )
    top = top.rename(columns={score_col: "top_q_or_p", odds_col: "top_odds"})
    top["enriched"] = (top["top_q_or_p"] < FDR_THR).astype(int)

    # depth + IC for top term
    top["go_depth"] = top["go_term"].astype(str).map(lambda t: _term_depth(t, godag))
    if ic_map is not None:
        top["go_ic"] = top["go_term"].astype(str).map(lambda t: ic_map.get(t, np.nan))
    else:
        top["go_ic"] = np.nan

    n_proto = int(top["proto"].nunique())
    n_enr = int(top["enriched"].sum())
    pct_enr = float(n_enr / n_proto) if n_proto else np.nan

    # odds ratio summary among enriched prototypes (more stable than mixing non-enriched)
    odds_enr = top.loc[top["enriched"] == 1, "top_odds"].to_numpy(dtype=float)
    odds_mu, odds_lo, odds_hi = _bootstrap_mean_ci(odds_enr, n_boot=N_BOOT, seed=SEED + hash(method) % 10_000)

    # GO depth/IC: take medians among enriched; fallback to all if no enriched
    depth_enr = top.loc[top["enriched"] == 1, "go_depth"].to_numpy(dtype=float)
    ic_enr = top.loc[top["enriched"] == 1, "go_ic"].to_numpy(dtype=float)
    if np.isfinite(depth_enr).sum() == 0:
        depth_enr = top["go_depth"].to_numpy(dtype=float)
    if np.isfinite(ic_enr).sum() == 0:
        ic_enr = top["go_ic"].to_numpy(dtype=float)

    rows.append({
        "method": method,
        "label": cfg.get("label", method),
        "n_prototypes": n_proto,
        f"% enriched (q<{FDR_THR})": 100.0 * pct_enr if np.isfinite(pct_enr) else np.nan,
        "mean OR (enriched)": odds_mu,
        "OR 95% CI low": odds_lo,
        "OR 95% CI high": odds_hi,
        "median GO depth (top term)": float(np.nanmedian(depth_enr)) if depth_enr.size else np.nan,
        "median IC (top term)": float(np.nanmedian(ic_enr)) if ic_enr.size else np.nan,
    })

    per_method_details[method] = top

q3_table = pd.DataFrame(rows)

# nicer formatting + stable order if you have KEEP_METHODS or similar
if "KEEP_METHODS" in globals():
    q3_table["method"] = pd.Categorical(q3_table["method"], categories=KEEP_METHODS, ordered=True)
    q3_table = q3_table.sort_values("method")

# compact CI column
if not q3_table.empty:
    q3_table["mean OR ±95%CI"] = q3_table.apply(
        lambda r: f"{r['mean OR (enriched)']:.2f} [{r['OR 95% CI low']:.2f}, {r['OR 95% CI high']:.2f}]"
        if np.isfinite(r["mean OR (enriched)"]) else "nan",
        axis=1
    )

display_cols = [
    "label",
    "n_prototypes",
    f"% enriched (q<{FDR_THR})",
    "mean OR ±95%CI",
    "median GO depth (top term)",
    "median IC (top term)",
]
display(q3_table[display_cols])



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

EPS = 1e-12
MIN_SEGS_PER_PROTEIN = 3  # filter unstable proteins

# Use only methods that have prototype labels available
METHOD_ORDER = [m for m in METHODS.keys() if m in method_proto_dense]
METHOD_LABEL = {m: METHODS[m].get("label", m) for m in METHOD_ORDER}

# Stable color map by method label
if "COLOR" not in globals():
    COLOR = {METHOD_LABEL[m]: f"C{i}" for i, m in enumerate(METHOD_ORDER)}

def runs_from_labels(y: np.ndarray, drop_neg: bool = True) -> np.ndarray:
    """
    Run labels from residue-level labels.
    Example: [1,1,1, 5,5, 2] -> [1,5,2]
    If drop_neg=True, removes labels <0 before run extraction (recommended).
    """
    y = np.asarray(y, dtype=int)
    if y.size == 0:
        return np.array([], dtype=int)

    if drop_neg:
        y = y[y >= 0]
        if y.size == 0:
            return np.array([], dtype=int)

    change = np.where(y[1:] != y[:-1])[0] + 1
    starts = np.r_[0, change]
    return y[starts]

def shannon_entropy(p: np.ndarray, eps: float = 1e-12) -> float:
    p = p[p > 0]
    if p.size == 0:
        return float("nan")
    return float(-np.sum(p * np.log(p + eps)))

def protein_proto_usage_metrics(method: str) -> pd.DataFrame:
    """
    For each protein b:
      run_labels = prototype id per segment/run in the residue prototype sequence
      c[b,k] = #runs assigned to prototype k
      p[b,k] = normalized counts
      top1_share = max_k p[b,k]
      Neff = exp(H(p[b]))
    """
    rows = []
    for prot in common_proteins:
        y_proto = method_proto_dense[method].get(prot, None)
        if y_proto is None or len(y_proto) == 0:
            continue

        run_labels = runs_from_labels(y_proto, drop_neg=True)
        n_seg = int(run_labels.size)
        if n_seg < MIN_SEGS_PER_PROTEIN:
            continue

        c = Counter(map(int, run_labels.tolist()))
        counts = np.array(list(c.values()), dtype=float)
        p = counts / counts.sum()

        top1 = float(np.max(p))
        Neff = float(np.exp(shannon_entropy(p, eps=EPS)))
        n_used = int(len(c))

        rows.append({
            "method": method,
            "label": METHOD_LABEL[method],
            "protein": prot,
            "n_segments": n_seg,
            "n_protos_used": n_used,
            "top1_share": top1,
            "Neff": Neff,
        })
    return pd.DataFrame(rows)

# ---- build per-protein table across methods ----
usage_df = pd.concat([protein_proto_usage_metrics(m) for m in METHOD_ORDER], ignore_index=True)
display(usage_df.head())

# ---------------------------
# Plot: quantile bars (horizontal)
# ---------------------------
def quantile_bars_ax(
    df: pd.DataFrame,
    col: str,
    ax,
    *,
    title: str,
    xlabel: str,
    q_outer=(0.10, 0.90),
    q_inner=(0.25, 0.75),
    min_n=20,
    xlim=None,
):
    methods = [m for m in METHOD_ORDER if METHOD_LABEL[m] in df["label"].unique()]
    y_pos = np.arange(len(methods))

    for i, m in enumerate(methods):
        lab = METHOD_LABEL[m]
        x = pd.to_numeric(df.loc[df["label"] == lab, col], errors="coerce").dropna().to_numpy(float)
        if x.size < min_n:
            continue

        qlo, qhi = np.quantile(x, q_outer)
        q1, q3 = np.quantile(x, q_inner)
        med = np.median(x)

        c = COLOR.get(lab, None)
        ax.plot([qlo, qhi], [i, i], linewidth=1.4, color=c, solid_capstyle="round")
        ax.plot([q1, q3], [i, i], linewidth=7.0, color=c, alpha=0.85, solid_capstyle="round")
        ax.scatter([med], [i], s=36, color=c, edgecolor="white", linewidth=0.7, zorder=3)

        ax.text(
            0.99, i, f"n={x.size}",
            transform=ax.get_yaxis_transform(),
            ha="right", va="center", fontsize=8, alpha=0.75
        )

    ax.set_yticks(y_pos)
    ax.set_yticklabels([METHOD_LABEL[m] for m in methods])
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.grid(True, axis="x", alpha=0.18)
    ax.grid(False, axis="y")
    if xlim is not None:
        ax.set_xlim(*xlim)

# ---- Replacement figure (instead of old proto reuse) ----
fig, (axA, axB) = plt.subplots(1, 2, figsize=(9.6, 3.4), gridspec_kw={"wspace": 0.35})

quantile_bars_ax(
    usage_df, "Neff", axA,
    title="A. Prototype usage breadth per protein",
    xlabel="Neff = exp(H(p)) (effective # prototypes used)",
    xlim=None,
)

quantile_bars_ax(
    usage_df, "top1_share", axB,
    title="B. Prototype dominance per protein",
    xlabel="top-1 share = max_k p[b,k]",
    xlim=(0, 1.0),
)
axB.set_yticks([])

fig.suptitle("Prototype usage within proteins (segments = runs of prototypes)", y=1.03, fontsize=13)
plt.tight_layout()
plt.show()

usage_summary = (
    usage_df.groupby("label")
    .agg(
        n_proteins=("protein", "nunique"),
        segs_per_protein_mean=("n_segments", "mean"),
        Neff_mean=("Neff", "mean"),
        Neff_median=("Neff", "median"),
        top1_mean=("top1_share", "mean"),
        top1_median=("top1_share", "median"),
        protos_used_mean=("n_protos_used", "mean"),
    )
    .reset_index()
)
display(usage_summary)
